# Justifying the S4D Model: A Comparison Against Ten Classical and Deep-Learning Baselines

This notebook trains eleven classifiers on the same GalaxyMNIST morphology classification
task and compares them on accuracy, inference speed, model size, and a few other angles:
S4D, CNN, RNN, Transformer, Logistic Regression, Decision Tree, SVM, KNN, Naive Bayes,
Random Forest, and Bayesian Logistic Regression.

**S4D uses the real v3 training pipeline, now with the full accuracy-improvement
roadmap folded in:** RGB input, patch embedding (a modified Hilbert scan grouping
pixels into patches instead of scanning them individually), mean pooling, a 3rd
S4 layer, and a bigger `d_model`/`d_state` -- plus the grouped optimizer, warmup,
cosine-annealing-with-warm-restarts, gradient clipping, and rotation/flip
augmentation the recipe always had. CNN, RNN, and Transformer are capacity- and
input-matched to whatever S4D is currently configured as (see below), so the
comparison stays fair as that configuration changes; the seven classical models
(Logistic Regression, Decision Tree, SVM, KNN, Naive Bayes, Random Forest, and
Bayesian Logistic Regression) are unaffected by any of this -- they all take the
same flat feature vector regardless of how S4D/CNN/RNN/Transformer are configured.

**Every architecture knob lives in one place** (section 1's config cell):
`COLORED`, `S4D_STATE`, `S4D_NUM_LAYERS`, `S4D_PATCH_SIZE`, `S4D_POOLING`. To
try a different size -- `S4D_STATE = 192` instead of `256`, say -- change it
there and rerun the notebook top to bottom ("Restart and run all"); nothing
else needs editing. CNN's hidden width, RNN's/Transformer's `d_model`, and
S4D's own layers all read from those same variables at instantiation time.

**Section 7.5 validates each roadmap change on its own** before section 8
commits to training the fully-assembled version for real: a quick few-epoch
smoke test after adding RGB, then patch embedding, then mean pooling, then the
capacity bump, each printing its own validation accuracy so a change that
doesn't help is visible immediately rather than buried in one long final number.

**A fairness note:** the comparison in section 8 holds S4D to the *same*
`BASELINE_EPOCHS` (15) budget as every other model that trains iteratively --
CNN, RNN, Transformer, and Logistic Regression -- so the headline accuracy
numbers are directly comparable across those five, no epoch-count asterisk.
S4D still gets the training-recipe treatment it actually needs to train at
all; only the epoch *count* is matched, not the recipe itself. The other six
models (Decision Tree, SVM, KNN, Naive Bayes, Random Forest, Bayesian Logistic
Regression) have no notion of epochs at all -- each is a single `.fit()` call,
so there's no budget to match in the first place, the same way this notebook
already treated Decision Tree before the other five joined it. Section 16, at
the very end, takes the model this fair comparison validates and retrains it
from scratch at its full production epoch budget to produce the actual
checkpoint worth deploying.

**`HilbertScan` and `TakeLastTimestep` are vendored into this notebook**
(section 5.1) rather than imported from the cloned repo's `model/hilbert.py` /
`model/tlts.py` -- and `HilbertScan` here is a genuine modification (patch-level
scanning, not just pixel-level), not merely a copy. That means the patch-embedding
architecture change lives entirely in this notebook; the project's own
`model/hilbert.py`, used by the recurrent RISC-V model, is untouched. One
consequence: S4D's state dict is no longer directly interchangeable with the
recurrent `model.gclassifier.GalaxyClassifierS4D` -- section 16 has the details.

**This notebook still clones your `S4-Enhancement-Exploration` repo** (`python`
branch), for `model.functions.load_data` and the `galaxy_mnist` package it
depends on -- same requirement as the main training notebook, not a new dependency.


> ## Changes in this version (starting point: 80.35% test acc @ 400 epochs)
>
> Four changes, aimed at closing the gap to the 85-90% target, based on
> reading this notebook's own results rather than generic advice:
>
> 1. **`GalaxyClassifierS4DFast` can now wrap each S4D layer in a pre-norm
>    residual block** (`S4D_USE_NORM` / `S4D_USE_RESIDUAL`, section 1 -- both
>    default `True`). The original class had no normalization or residual
>    connections anywhere, unusual for a 3-layer stacked SSM and a plausible
>    reason the 400-epoch run plateaus hard after epoch ~300 rather than
>    still improving. `S4D_DROPOUT` (default `0.0`) is also new, left off
>    since the 15-epoch comparison run shows no overfitting signal yet.
> 2. **Section 7.5 gets two new ablation steps (5 and 6)** to smoke-test
>    norm+residual, and mean-vs-last pooling, the same cheap way this
>    notebook already validates every other architecture choice, before
>    section 16 spends a full run on either.
> 3. **`S4D_FINAL_EPOCHS` is 630, not 400.** The warm-restart schedule's own
>    cycle boundaries (10, 30, 70, 150, 310, 630) mean a 400-epoch run stops
>    90 epochs into an unfinished cycle, at a learning rate nowhere near
>    `eta_min` -- and the existing 400-epoch run's per-epoch table backs this
>    up: validation accuracy peaks at epoch 304 and then just oscillates
>    through epoch 400. 630 lets that cycle actually finish. Full reasoning
>    in section 16's intro.
> 4. **Nothing about sections 1-15's original results was deleted.** Their
>    outputs still show the original 80.35%-test-acc run (comparison-run
>    section included) until you re-run them -- useful as a before/after.
>    **To get new numbers: Runtime -> Restart session, then Runtime -> Run
>    all.** Everything reads from the section 1 config cell, so no other
>    manual edits are needed to pick up these changes.
>
> Two things intentionally *not* changed here, flagged instead of guessed at:
> patch size (worth trying smaller, e.g. `S4D_PATCH_SIZE=2`, to help the
> Smooth-Cigar/Edge-on-Disk confusion the production confusion matrix shows
> as the dominant error) and augmentation strength (only rotation/flip
> currently) -- both reasonable follow-ups once this round's results are in,
> but stacking too many untested changes into one run makes it hard to tell
> which change did what.

## 1. Setup

In [ ]:
# 1. Clone the repository (same as the main training notebook)
!git clone https://github.com/ahsan-c0ding/S4-Enhancement-Exploration.git
%cd S4-Enhancement-Exploration
!git checkout python

import os
import sys
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"Working directory set to: {current_dir}")

In [ ]:
!nvidia-smi

In [ ]:
# If you have a GPU, prefer installing the CUDA build of PyTorch -- this notebook trains
# S4D via the real v3 recipe (up to S4D_FINAL_EPOCHS epochs in section 16, 200 by default),
# which needs GPU to finish in a reasonable time.
# Refer to https://pytorch.org/get-started/locally/ for the exact command for your CUDA version, e.g.:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install torch torchvision --quiet
%pip install numpy matplotlib scikit-learn h5py tqdm seaborn torchinfo einops --quiet
%pip install transformers 'accelerate>=1.1.0' ipywidgets coloredlogs --quiet
%pip install git+https://github.com/mwalmsley/galaxy_mnist.git@c1fe9853a00bc34b2ff082585c6bb1654d34d239 --quiet

In [ ]:
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import joblib

In [ ]:
# Workaround for a bug in model/__init__.py: it reads model_params/model_weights.bin
# using a path relative to the current working directory ("../model_params/...")
# instead of relative to the model/ package itself. That only resolves correctly if
# cwd happens to be one level *inside* the repo when `model` gets imported -- but the
# cd above (same as the main training notebook) puts cwd at the repo root, one level
# too shallow, so `import model` fails with FileNotFoundError. This symlink makes the
# path resolve correctly regardless of cwd. The real fix belongs in model/__init__.py
# itself: make SRC_BIN / OUT_PTH relative to os.path.dirname(__file__), the same
# pattern already used for the sys.path.append two lines above them in that file.
parent_dir = os.path.dirname(current_dir)
_shim_path = os.path.join(parent_dir, "model_params")
_real_path = os.path.join(current_dir, "model_params")
if not os.path.exists(_shim_path) and os.path.exists(_real_path):
    os.symlink(_real_path, _shim_path)
    print(f"Symlinked {_shim_path} -> {_real_path}")
elif os.path.exists(_shim_path):
    print(f"{_shim_path} already present, leaving as-is")
else:
    print(f"WARNING: {_real_path} not found -- importing `model` will still fail")

In [ ]:
# Project modules -- same imports the main training notebook relies on.
# NOTE: HilbertScan and TakeLastTimestep are deliberately NOT imported from
# model.hilbert / model.tlts here -- section 5.1 defines local copies of both
# instead, so this notebook never needs to touch those project files.
from model.functions import load_data
from utils import set_pbar_style

warnings.filterwarnings("ignore")
set_pbar_style(bar_fill_color="#FFFFFF", text_color="#FFFFFF")
sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = [11, 6]

In [ ]:
RNG_SEED = 30485  # same seed as the main training notebook

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RNG_SEED)

CLASS_NAMES = ["Smooth Round", "Smooth Cigar", "Edge-on Disk", "Unbarred Spiral"]

# ============================================================================
# MODEL CONFIGURATION -- the knobs below are read by name everywhere else in
# the notebook, nothing is hardcoded a second time. To try a different S4D
# size, change S4D_STATE here and rerun from the top ("Restart and run all"):
# CNN's hidden width, RNN's/Transformer's d_model, and S4D's own d_model/
# d_state all derive from it, so every neural model stays at matched capacity
# without touching any other cell.
# ============================================================================
COLORED = True             # RGB (3ch) input instead of grayscale (1ch), for every model
IN_CHANNELS = 3 if COLORED else 1

S4D_STATE = 256             # d_model == d_state for S4D (this project's convention);
                            # CNN's hidden width and RNN's/Transformer's d_model match this too
S4D_NUM_LAYERS = 3          # stacked S4DConv layers
S4D_PATCH_SIZE = 4          # 64x64 image -> (64/S4D_PATCH_SIZE)^2 Hilbert-ordered patches
                            # (4 -> 256 patches; 1 -> the original 4096-pixel, no-patching scan)
S4D_POOLING = "last"        # "mean" (average over the sequence) or "last" (TakeLastTimestep)
                            # confirmed by Step 6 vs 7 -- keep this change

S4D_USE_NORM = False      # revised -- Steps 4-7 all favor this being off
S4D_USE_RESIDUAL = False  # revised, same evidence
S4D_DROPOUT = 0.2  # was 0.0 -- justified now: train_acc 0.9539 vs test_acc 0.8270,
                    # a much bigger gap than any earlier run showed
S4D_PATCH_EMBED = "conv"  # "linear" (Linear on a flattened raw patch) or "conv"
                             # (ConvPatchStem) -- see section 7.5 Step 8 before changing this

# Auto-namespaces every checkpoint dir below by whatever actually affects
# checkpoint compatibility -- change any of these and the next run gets its
# own fresh directory instead of trying (and failing) to resume from an
# architecturally-incompatible one, which is what happened going from
# S4D_PATCH_SIZE=2 to 4 with a fixed directory name.
import hashlib

# Every S4D_* knob EXCEPT the epoch-count ones -- those are the one dimension
# you resume *across* a change on, not fork a new directory for. (This is what
# broke going from S4D_FINAL_EPOCHS=630 to 693: the old tag formula swept in
# every S4D_-prefixed global with no exceptions, so that change alone pointed
# CHECKPOINT_DIR_FINAL at a directory that had never existed, and training
# restarted from epoch 0 instead of resuming the 630-epoch checkpoint.)
_S4D_DURATION_KNOBS = {"S4D_FINAL_EPOCHS", "S4D_COMPARISON_EPOCHS"}
s4d_config_snapshot = {
    k: v for k, v in sorted(globals().items())
    if k.startswith("S4D_") and k not in _S4D_DURATION_KNOBS
    and isinstance(v, (int, float, str, bool))
}
s4d_config_tag = hashlib.md5(repr(s4d_config_snapshot).encode()).hexdigest()[:10]
print("S4D config snapshot for this checkpoint tag:")
for k, v in s4d_config_snapshot.items():
    print(f"  {k} = {v!r}")
print(f"-> tag: {s4d_config_tag}")

S4D_BATCH_SIZE = 32         # S4D-specific training batch size (sections 8, 16)
ABLATION_EPOCHS = 12         # quick smoke-test length for section 7.5's incremental checks

os.makedirs("model_comparison_outputs", exist_ok=True)

print(f"Using RNG seed: {RNG_SEED}")
print(f"Using device: {DEVICE}")
print(f"COLORED={COLORED}  S4D_STATE={S4D_STATE}  S4D_NUM_LAYERS={S4D_NUM_LAYERS}  "
      f"S4D_PATCH_SIZE={S4D_PATCH_SIZE}  S4D_POOLING={S4D_POOLING!r}")
print(f"S4D_USE_NORM={S4D_USE_NORM}  S4D_USE_RESIDUAL={S4D_USE_RESIDUAL}  S4D_DROPOUT={S4D_DROPOUT}  "
      f"S4D_PATCH_EMBED={S4D_PATCH_EMBED!r}")

## 2. Load and Preprocess GalaxyMNIST

Uses `model.functions.load_data` directly -- the same data every model in this
notebook, including S4D, is trained and evaluated on.

In [ ]:
X, y_onehot, y = load_data(root="./data", download=True, train=True, colored=COLORED)
NUM_CLASSES = y_onehot.shape[1]

X_test, y_test_onehot, y_test = load_data(root="./data", download=True, train=False, colored=COLORED)

print(f"X shape: {X.shape}, y shape: {y.shape}, y_onehot shape: {y_onehot.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
BATCH_SIZE = 64

# One split, shared by every model in this notebook (same random_state as the main
# training notebook, so this is the same train/val partition it uses).
x_train, x_val, y_train_onehot, y_val_onehot, y_train, y_val = train_test_split(
    X, y_onehot, y, test_size=0.2, random_state=RNG_SEED, stratify=y
)

# Integer-label loaders: used for training CNN/RNN/Transformer, and for evaluating
# ALL six models (including S4D) on a single, consistent metric-computation path.
train_ds = TensorDataset(x_train, y_train)
val_ds = TensorDataset(x_val, y_val)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

## 3. Hilbert Curve Utilities (for RNN and Transformer only)

S4D doesn't need this section -- `GalaxyClassifierS4DFast` does its own Hilbert
scan internally via the notebook-local `HilbertScan` defined in section 5.1. This section is only for RNN and
Transformer, which get a *patch*-level Hilbert ordering (8x8 grid of 8x8-pixel
patches, 64 tokens) rather than the full 4096-pixel sequence S4D operates on --
full self-attention over 4096 raw pixels would need tens of GB per batch just for
the attention scores, so patch tokenization is what keeps them trainable at all.

In [ ]:
def hilbert_d2xy(order, d):
    """Map a distance d along a Hilbert curve of the given order to (x, y) grid coordinates."""
    x = y = 0
    t = d
    s = 1
    while s < (1 << order):
        rx = 1 & (t // 2)
        ry = 1 & (t ^ rx)
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        x += s * rx
        y += s * ry
        t //= 4
        s *= 2
    return x, y


def hilbert_curve_order(grid_size):
    """Return the (row, col) visiting order of a Hilbert curve over a grid_size x grid_size grid."""
    order = int(np.log2(grid_size))
    assert 2 ** order == grid_size, "grid_size must be a power of 2"
    coords = np.array([hilbert_d2xy(order, d) for d in range(grid_size * grid_size)])
    return coords[:, 1], coords[:, 0]  # rows, cols


patch_rows, patch_cols = hilbert_curve_order(8)
PATCH_HILBERT_IDX = torch.tensor(patch_rows * 8 + patch_cols, dtype=torch.long)


def hilbert_patch_tokens(images, patch_size=8, hilbert_order=PATCH_HILBERT_IDX):
    """Chop the image into patch_size x patch_size patches and flatten each into a token,
    visiting patches in Hilbert order so nearby patches stay nearby in the sequence."""
    b, c, h, w = images.shape
    p = patch_size
    patches = images.unfold(2, p, p).unfold(3, p, p)  # (B, C, H/p, W/p, p, p)
    gh, gw = patches.shape[2], patches.shape[3]
    patches = patches.contiguous().view(b, c, gh * gw, p * p)
    patches = patches.permute(0, 2, 1, 3).reshape(b, gh * gw, c * p * p)
    return patches[:, hilbert_order, :]

## 4. Model-Specific Data Adapters

In [ ]:
def adapt_cnn(images):
    """CNN sees the native (B, 1, 64, 64) grid."""
    return images


def adapt_s4d(images):
    """GalaxyClassifierS4DFast does its own Hilbert scan internally (via the
    notebook-local HilbertScan from section 5.1) -- so, like the CNN, it just
    wants the raw image."""
    return images


def adapt_patches(images):
    """RNN and Transformer both take the same 64-token Hilbert-ordered patch sequence."""
    return hilbert_patch_tokens(images, patch_size=8)


def adapt_flat_numpy(images):
    """Logistic Regression and Decision Tree want a flat feature vector per sample.
    Pixel order doesn't matter to either (they treat each column independently),
    so a plain flatten is equivalent to a Hilbert-ordered one here, and cheaper."""
    return images.view(images.size(0), -1).numpy()

## 5. Model Zoo

### 5.1 S4D -- the Real v3 Training Pipeline

`S4DConv` below is copied directly from the main training notebook -- the FFT-based
parallel convolution, not reimplemented. `GalaxyClassifierS4DFast` has since
diverged from `model.gclassifier.GalaxyClassifierS4D` (the recurrent, RISC-V-portable
version): it now supports patch embedding, mean pooling, and a configurable
number of layers (section 7.5 validates each of these choices), none of which
the recurrent model implements. **That means state dicts trained here no longer
load into the recurrent model directly** -- porting these gains back to
RISC-V would mean updating `model/s4d_recurrent.py` and `model/gclassifier.py`
to match, which is outside this notebook's scope. With `S4D_PATCH_SIZE=1` and
`S4D_POOLING="last"`, this class reduces to exactly the original architecture
(one pixel per token, take-last pooling) -- so the divergence is a choice made
here via config, not a hardcoded rewrite.

The short version of why `S4DConv` exists (full diagnosis is in the main
notebook): the recurrent `S4D` layer steps through every timestep with a Python
loop, which is fine for a single-sample, no-autograd inference loop on RISC-V,
but strictly sequential and un-parallelizable for GPU training. `S4DConv`
computes the exact same math as one FFT-based global convolution instead --
about an 18x speedup on the original sequence length, with identical outputs
given identical weights.

#### Hilbert Scan & Take-Last Timestep (vendored *and modified* in this notebook)

`HilbertScan` and `TakeLastTimestep` normally live in the cloned repo, at
`model/hilbert.py` and `model/tlts.py`. `TakeLastTimestep` is copied verbatim
below -- same last-timestep extraction, no changes. `HilbertScan` is a real
modification, not just a copy: the original scans individual *pixels* (a fixed
4096-length sequence); this version scans *patches* (`S4D_PATCH_SIZE x
S4D_PATCH_SIZE` blocks of pixels, each flattened into one token), with the
Hilbert curve computed over the patch grid instead of the pixel grid. That's
what makes the 256-length sequence from section 7.5's roadmap possible: fewer,
richer tokens instead of thousands of individual pixel values. `patch_size=1`
recovers the original pixel-level behavior exactly (each "patch" is a single
pixel), so nothing about the original algorithm was thrown away -- it's the
`patch_size=1` special case of this more general version.

This lives in the notebook rather than in `model/hilbert.py` itself so that
experimenting with it for this comparison never touches the project's files --
the original pixel-level scan the recurrent RISC-V model depends on stays
completely untouched.

In [ ]:
class HilbertScan(nn.Module):
    """Reorders PATCHES of a (B, C, H, W) image according to a Hilbert curve
    computed over the patch grid, and returns (B, num_patches, C*patch_size^2)
    -- each patch flattened into one token. A genuine modification of
    model/hilbert.py (which scans individual pixels, i.e. patch_size=1):
    grouping pixels into patches first is what cuts the sequence length from
    4096 down to (image_size/patch_size)^2, making a 3rd S4 layer and a much
    bigger d_model affordable. Modified here, in the notebook, rather than in
    model/hilbert.py itself -- the original pixel-level scan the recurrent
    RISC-V model depends on stays untouched."""

    def __init__(self, image_size=64, patch_size=1):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size      # patches per side
        self.num_patches = self.grid_size ** 2
        self.register_buffer("indices", self._get_hilbert_indices(self.grid_size))

    @staticmethod
    def _rot(s, x, y, rx, ry):
        if ry == 0:                  # Bottom half of the current square
            if rx == 1:               # Bottom-right quadrant
                x = s - 1 - x         # Reflect over diagonal
                y = s - 1 - y
            x, y = y, x                # Swap x and y for 90-degree rotation
        return x, y

    def _d2xy(self, n, d):
        """Convert a 1D Hilbert curve distance to 2D (x, y) coordinates."""
        x = y = 0
        t = d
        s = 1
        while s < n:
            rx = (t // 2) & 1
            ry = (t ^ rx) & 1
            x, y = self._rot(s, x, y, rx, ry)
            x += s * rx
            y += s * ry
            t //= 4
            s *= 2
        return x, y

    def _get_hilbert_indices(self, grid_size):
        indices = []
        for d in range(grid_size * grid_size):
            x, y = self._d2xy(grid_size, d)
            indices.append(y * grid_size + x)
        return torch.LongTensor(indices)

    def forward(self, x):
        # x: (B, C, H, W) -> (B, num_patches, C*patch_size^2), patches in Hilbert order.
        # patch_size=1 is exactly the original per-pixel scan: num_patches=H*W,
        # each "patch" is one pixel's channel values.
        B, C, H, W = x.shape
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)                # (B, C, gh, gw, p, p)
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()   # (B, gh, gw, C, p, p)
        patches = patches.view(B, self.num_patches, C * p * p)     # (B, num_patches, C*p*p)
        return patches[:, self.indices, :]


class TakeLastTimestep(nn.Module):
    """Extracts the final timestep of a (B, L, D) sequence -> (B, D). Vendored
    from model/tlts.py -- the last position has been updated by every prior
    timestep, so it serves as a compressed summary of the whole sequence."""

    def forward(self, x):
        return x[:, -1, :]

In [ ]:
import math
from einops import repeat


class S4DConv(nn.Module):
    """Fast (FFT-based, parallel-convolution) S4D layer, for TRAINING only."""

    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1, transposed=True, lr=None):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed

        log_dt = torch.rand(self.h) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        log_A_real = torch.log(0.5 * torch.ones(self.h, self.n // 2))
        A_imag = math.pi * repeat(torch.arange(self.n // 2), 'n -> h n', h=self.h)
        C_init = torch.randn(self.h, self.n // 2, dtype=torch.cfloat)

        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)

        self.C = nn.Parameter(torch.view_as_real(C_init))
        self.D = nn.Parameter(torch.randn(self.h))

    def register(self, name, tensor, lr=None):
        # identical to model/s4d_recurrent.py's register() on purpose -- keeps
        # naming conventions consistent even though this model's overall
        # state_dict no longer matches the recurrent model's (see 5.1's note).
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            optim = {"weight_decay": 0.0}
            if lr is not None:
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

    def forward(self, u):
        # u: (B, H, L) if transposed else (B, L, H)
        if not self.transposed:
            u = u.transpose(-1, -2)
        L = u.size(-1)

        dt = torch.exp(self.log_dt)
        C = torch.view_as_complex(self.C)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag

        dtA = A * dt.unsqueeze(-1)
        # Vectorized kernel generation -- no Python loop over L.
        K_exp = torch.exp(dtA.unsqueeze(-1) * torch.arange(L, device=u.device))
        C_tilde = C * (torch.exp(dtA) - 1.) / A
        k = 2 * torch.einsum('hn, hnl -> hl', C_tilde, K_exp).real

        # Whole-sequence convolution via FFT, in one shot, fully parallel.
        k_f = torch.fft.rfft(k, n=2 * L)
        u_f = torch.fft.rfft(u, n=2 * L)
        y = torch.fft.irfft(u_f * k_f, n=2 * L)[..., :L]

        y = y + u * self.D.unsqueeze(-1)

        if not self.transposed:
            y = y.transpose(-1, -2)
        return y, None


class ConvPatchStem(nn.Module):
    """Alternative to a bare `Linear` patch embedding. A raw `patch_size x
    patch_size` block flattened and passed through one `Linear` (the
    `patch_embed="linear"` path below) only ever sees pixels *inside* its own
    patch -- it structurally cannot represent a feature that straddles a patch
    boundary, like a faint dust lane a couple of pixels wide. This stem runs a
    3x3, stride-1 conv first (every position starts mixing in its immediate
    neighbors, patch boundaries included), then a stride-`patch_size` conv
    that downsamples to the same target grid the linear path produces while
    finishing the projection to `d_model`. Same output shape as the linear
    path either way -- (B, d_model, grid, grid) before Hilbert-ordering --
    so it's a drop-in alternative, not a different sequence length.

    The CNN baseline elsewhere in this notebook gets exactly this kind of
    local receptive field for free from its conv layers; the S4D path never
    had it until now, which section 7.5's Step 8 exists to check the value
    of, given patch_size alone (Step 2, and section 1's patch_size 4->2
    change) hasn't moved the Smooth-Cigar/Edge-on-Disk confusion much."""

    def __init__(self, in_channels, d_model, patch_size):
        super().__init__()
        mid_channels = max(in_channels * 8, 32)
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
            nn.Conv2d(mid_channels, d_model, kernel_size=patch_size, stride=patch_size),
        )

    def forward(self, x):
        return self.net(x)   # (B, d_model, H/patch_size, W/patch_size)


class GalaxyClassifierS4DFast(nn.Module):
    """Training-only S4D classifier. Every architecture choice from the roadmap
    (RGB, patch embedding, mean pooling, layer count, capacity) is a constructor
    argument rather than a hardcoded rewrite -- with num_layers=2, patch_size=1,
    pooling="last", this is exactly the original architecture; every other
    combination is a genuine variant, not a from-scratch redesign. Section 7.5
    validates each argument's effect individually before section 8 trains the
    fully-assembled version end to end.

    use_norm / use_residual / dropout (all off by default, so every existing
    call above -- steps 1-4 -- is completely unaffected) wrap each S4D layer in
    a pre-norm residual block: h = h + Dropout(Act(S4D(LayerNorm(h)))) instead
    of the bare h = Act(S4D(h)) this class started with. That block shape is
    the standard one in the S4/S4D papers' own stacked models; this class
    originally omitted it to save memory (see section 13's writeup), which is
    a real cost worth re-checking now that S4D_NUM_LAYERS=3 stacks three of
    these with nothing to stabilize the stack or shorten the gradient path
    back to layer 1. Steps 5-6 in section 7.5 test whether it earns its keep.

    patch_embed="conv" (default remains "linear", so every existing call is
    still unaffected) swaps the bare `Linear` patch projection for
    `ConvPatchStem` above -- same token count and d_model either way, just a
    different way of building each token. See `ConvPatchStem`'s docstring for
    why this is a different lever than shrinking patch_size further."""

    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True,
                 num_layers=2, patch_size=1, pooling="last",
                 use_norm=False, use_residual=False, dropout=0.0,
                 patch_embed="linear"):
        super().__init__()
        self.hilbert_channels = 1 if not colored else 3
        self.patch_size = patch_size
        self.pooling = pooling
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.patch_embed = patch_embed

        if patch_embed == "linear":
            self.hilbert_scan = HilbertScan(image_size=64, patch_size=patch_size)
            patch_dim = self.hilbert_channels * patch_size * patch_size
            self.uproject = nn.Linear(patch_dim, d_model)
            self.conv_stem = None
        elif patch_embed == "conv":
            # HilbertScan(patch_size=1) over the stem's own output grid: the
            # stem has already done the downsampling-and-projection job the
            # Linear path does, so there's nothing left for uproject to do.
            self.conv_stem = ConvPatchStem(self.hilbert_channels, d_model, patch_size)
            self.hilbert_scan = HilbertScan(image_size=64 // patch_size, patch_size=1)
            self.uproject = nn.Identity()
        else:
            raise ValueError(f"patch_embed must be 'linear' or 'conv', got {patch_embed!r}")

        self.s4_layers = nn.ModuleList([
            S4DConv(d_model=d_model, d_state=s4_state, transposed=False)
            for _ in range(num_layers)
        ])
        self.acts = nn.ModuleList([nn.GELU() for _ in range(num_layers)])
        # One LayerNorm per block, pre-norm style (normalize the block's input,
        # not its output) -- matches the S4/S4D papers' reference block and is
        # generally the more stable choice for optimizing a multi-layer stack.
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)]) if use_norm else None
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        if pooling == "last":
            self.take_last = TakeLastTimestep()
        elif pooling == "mean":
            self.take_last = None   # pooling done inline in forward() -- no params needed
        else:
            raise ValueError(f"pooling must be 'last' or 'mean', got {pooling!r}")

        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=False):
        B, C, H, W = x.shape
        assert H == 64 and W == 64, "Expected 64x64"
        assert C == self.hilbert_channels, f"Expected {self.hilbert_channels} channels"

        if self.patch_embed == "conv":
            feat = self.conv_stem(x)          # (B, d_model, 64/patch_size, 64/patch_size)
            x_seq = self.hilbert_scan(feat)    # (B, num_patches, d_model)
            h = self.uproject(x_seq)           # Identity -- stem already projected to d_model
        else:
            x_seq = self.hilbert_scan(x)
            h = self.uproject(x_seq)

        for i, (s4_layer, act) in enumerate(zip(self.s4_layers, self.acts)):
            residual = h
            h_in = self.norms[i](h) if self.use_norm else h
            h_out, _ = s4_layer(h_in)
            h_out = act(h_out)
            h_out = self.drop(h_out)
            h = residual + h_out if self.use_residual else h_out

        pooled = h.mean(dim=1) if self.take_last is None else self.take_last(h)
        logits = self.fc(pooled)

        if return_logits:
            return logits
        return self.softmax(logits)


class LogitsAdapter(nn.Module):
    """GalaxyClassifierS4DFast needs return_logits=True to hand back raw logits --
    this wrapper makes it a plain model(x)->logits callable, so it can drop into
    the same evaluate_nn_model / measure_inference_speed / train_nn_model
    functions every other model in this notebook uses, with no special-casing
    in the harness itself."""

    def __init__(self, base):
        super().__init__()
        self.base = base

    def forward(self, x):
        return self.base(x, return_logits=True)

### 5.2 CNN

The "obvious" architecture for image data -- convolutions exploit 2D spatial locality
directly, without needing any Hilbert-curve trick to preserve it. Its final hidden
layer width matches `S4D_STATE`, same as RNN's and Transformer's `d_model` below --
the conv feature-extractor itself is untouched, only the dense bottleneck right
before classification is capacity-matched.

In [ ]:
class GalaxyCNN(nn.Module):
    def __init__(self, num_classes=4, in_channels=1, hidden_dim=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),  # 64->32
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),           # 32->16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),           # 16->8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, hidden_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

### 5.3 RNN (LSTM)

Processes the Hilbert-ordered patch sequence one token at a time, carrying state
forward. This is the classic sequence-modelling approach S4D was designed to improve
on -- mainly around gradient flow over long sequences and lack of parallel training.

In [ ]:
class GalaxyRNN(nn.Module):
    def __init__(self, num_classes=4, patch_dim=64, d_model=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(patch_dim, d_model)
        self.lstm = nn.LSTM(d_model, d_model, num_layers=num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.in_proj(x)
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])

### 5.4 Transformer

A small transformer encoder with a learned CLS token, over the same patch sequence
as the RNN. Deliberately fed *patch* tokens (64 of them) rather than raw pixels
(4096 of them): full self-attention is O(n^2) in sequence length, so at n=4096 the
attention score matrix alone would need tens of gigabytes per batch. S4D's FFT
convolution doesn't have this problem -- one of the findings this notebook is set
up to demonstrate.

In [ ]:
class GalaxyTransformer(nn.Module):
    def __init__(self, num_classes=4, patch_dim=64, d_model=64, nhead=4, num_layers=2,
                 dim_ff=128, dropout=0.1, num_patches=64):
        super().__init__()
        self.in_proj = nn.Linear(patch_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
                                                dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        b = x.size(0)
        x = self.in_proj(x)
        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.encoder(x)
        return self.classifier(self.norm(x[:, 0]))

### 5.5 Classical ML Baselines

Seven models here take the same flat, per-pixel feature vector (`adapt_flat_numpy`
in section 9 -- pixel order doesn't matter to any of them, so a plain flatten is as
good as a Hilbert-ordered one, and cheaper). All seven get added to `results` /
`predictions` the same way S4D/CNN/RNN/Transformer do, via the shared
`collect_sklearn_model_metrics` (section 6) instead of `collect_model_metrics`.

- **Logistic Regression** is trained with `SGDClassifier(loss="log_loss")`, called
  in mini-batches over the same 15 passes over the data as CNN/RNN/Transformer get --
  plain logistic regression, just optimized via SGD instead of a closed-form solve,
  so it gives a genuine, comparable "training accuracy per epoch" curve.
- **Decision Tree**, **SVM**, **KNN**, **Naive Bayes**, and **Random Forest** have no
  notion of epochs -- each is a single `.fit()` call. Reported for final accuracy,
  size, and speed; `train_time_s` isn't controlled by the same epoch budget as the
  five iterative models above, and what `params` means varies by model (section 10's
  note spells out each one):
  - **SVM** uses an RBF kernel (`sklearn.svm.SVC`) -- the actual kernel trick, not
    just a linear boundary (Logistic Regression already covers that case). The
    slowest of these five to fit at this feature count (12,288 flattened pixels) --
    a few minutes on a Colab CPU is normal, libsvm's solver doesn't parallelize
    across cores the way the tree-based models below do.
  - **KNN** (`k=5`, sklearn's default) has no learned weights at all -- `.fit()`
    just stores the training set, so all of its "work" happens at predict time.
  - **Naive Bayes** uses `GaussianNB` -- pixel intensities are continuous, not
    counts, so not `MultinomialNB`/`BernoulliNB`.
  - **Random Forest** is `DecisionTreeClassifier` bagged over `n_estimators=100`
    trees -- ensembling regularizes it enough that it doesn't need the single
    tree's `max_depth=16` cap.
- **Bayesian Logistic Regression** (section 5.6, its own subsection since it needs
  real code rather than an off-the-shelf `sklearn` class) also has no epoch loop --
  the MAP fit's `lbfgs` solver iterates internally, same as any other single
  `.fit()` call above.

### 5.6 Bayesian Logistic Regression (Laplace Approximation)

The other six classical baselines are one-line `sklearn` classes; this one needs
actual code, since `sklearn` doesn't ship a Bayesian logistic regression.

**What "Bayesian" means here, concretely:** fit the same MAP point estimate
`sklearn.linear_model.LogisticRegression`'s L2 penalty already gives you --
that penalty term *is* the log of a zero-mean Gaussian prior on the weights,
precision `1/C` -- and then approximate the posterior around that point as
Gaussian too (a Laplace approximation), using the curvature of the
log-posterior at the MAP solution as the posterior precision. The exact
curvature for softmax regression is a `(n_classes * n_features)^2` Hessian --
here, `(4 * 12288)^2`, on the order of 10^9 entries, not invertible at this
scale. `BayesianLogisticRegression` below uses the standard tractable
simplification: only the diagonal (per-weight variance, no cross-feature or
cross-class covariance). That diagonal is still the *exact* per-weight
curvature of the true multinomial log-likelihood, `sum_i x_ij^2 p_i(c)(1-p_i(c))`
plus the prior's `1/C` -- the approximation is dropping the off-diagonal terms,
not the per-weight ones.

**Why bother, if the point estimate is identical to plain `LogisticRegression`:**
prediction isn't just softmax(MAP logits) here. Each class's logit gets shrunk
toward 0 first, by an amount that grows with that logit's *posterior variance*
at that specific input (Bishop, *Pattern Recognition and Machine Learning*,
S4.5.2 -- the probit approximation to a sigmoid-Gaussian convolution). A point
far from where training data was dense gets a less confident prediction than
the same point would get from plain point-estimate Logistic Regression, purely
because the model is less sure of its weights out there -- that's the entire
practical difference Bayesian treatment buys over a point estimate, and it's
the kind of thing the noise-robustness check (section 12) should actually be
able to see.

In [ ]:
class BayesianLogisticRegression:
    """Laplace-approximated Bayesian multinomial logistic regression -- see
    section 5.6 for the derivation. `.fit()`/`.predict()`/`.predict_proba()`
    mirror the sklearn estimator interface closely enough to drop into
    `collect_sklearn_model_metrics` unchanged, without actually subclassing
    `BaseEstimator` (nothing here needs `get_params`/`set_params`)."""

    def __init__(self, C=1.0, max_iter=200, random_state=None):
        self.C = C
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        # C here is the exact same regularization strength LogisticRegression's
        # C already means -- Gaussian prior precision 1/C on every weight.
        self.map_ = LogisticRegression(
            C=self.C, solver="lbfgs", max_iter=self.max_iter,
            random_state=self.random_state,
        )
        self.map_.fit(X, y)
        self.coef_ = self.map_.coef_            # (n_classes, n_features)
        self.intercept_ = self.map_.intercept_  # (n_classes,)

        # Exact diagonal of the multinomial log-likelihood's Hessian at the MAP
        # point -- H_diag[j, c] = sum_i x_ij^2 * p_i(c) * (1 - p_i(c)) -- plus
        # the prior's contribution (1/C per weight, from d^2/dw^2 of the L2
        # term w^2/(2C)). One matmul, (n_features, n_samples) @ (n_samples,
        # n_classes) -> (n_features, n_classes); no per-class loop needed.
        P = self.map_.predict_proba(X)                              # (n, K)
        H_diag = (X ** 2).T @ (P * (1 - P)) + 1.0 / self.C           # (d, K)
        self.weight_var_ = (1.0 / H_diag).T                         # (K, d)
        return self

    def _moderated_logits(self, X):
        mu = X @ self.coef_.T + self.intercept_        # MAP logits, (n, K)
        sigma2 = (X ** 2) @ self.weight_var_.T          # predictive variance, (n, K)
        kappa = 1.0 / np.sqrt(1.0 + np.pi * sigma2 / 8.0)   # Bishop eq. 4.154
        return kappa * mu

    def predict_proba(self, X):
        logits = self._moderated_logits(X)
        logits = logits - logits.max(axis=1, keepdims=True)   # numerically stable softmax
        exp = np.exp(logits)
        return exp / exp.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

## 6. Unified Training & Evaluation Harness

Every model, including S4D, gets its final `train_acc` / `val_acc` / `test_acc` /
macro-F1 / latency / throughput / noise-robustness numbers from the *same* functions
below, run against the *same* `train_loader` / `val_loader` / `test_loader`. Only the
training procedure differs per model -- how each one gets to its final weights isn't
forced to be identical, but how every one of them is measured afterward is.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


def torch_model_size_mb(model, path):
    torch.save(model.state_dict(), path)
    return os.path.getsize(path) / 1e6


def sklearn_model_size_mb(model, path):
    joblib.dump(model, path)
    return os.path.getsize(path) / 1e6


def measure_inference_speed(model, sample_batch, device, n_repeats=30):
    """Single-sample latency (ms) and batched throughput (samples/sec)."""
    model.eval()
    single = sample_batch[:1].to(device)
    batch = sample_batch.to(device)

    with torch.no_grad():
        for _ in range(5):  # warmup, avoids timing lazy CUDA kernel compilation
            model(single)
        if device == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            model(single)
            if device == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
        latency_ms = float(np.mean(times) * 1000)

        t0 = time.perf_counter()
        for _ in range(10):
            model(batch)
        if device == "cuda":
            torch.cuda.synchronize()
        throughput = 10 * batch.size(0) / (time.perf_counter() - t0)

    return latency_ms, float(throughput)


def evaluate_nn_model(model, adapter, loader, device):
    model.eval()
    correct, total = 0, 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            x, labels = adapter(images).to(device), labels.to(device)
            logits = model(x)
            preds = logits.argmax(-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
    acc = correct / total
    f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return acc, f1, np.array(all_targets), np.array(all_preds)


def evaluate_with_noise(model, adapter, loader, device, noise_std=0.15):
    """Accuracy under additive Gaussian pixel noise -- a cheap proxy for robustness."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            noisy = (images + torch.randn_like(images) * noise_std).clamp(0, 1)
            x, labels = adapter(noisy).to(device), labels.to(device)
            preds = model(x).argmax(-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


def train_nn_model(model, adapter, train_loader, val_loader, epochs, lr, device, model_name=""):
    """Generic training loop, used for CNN / RNN / Transformer. S4D uses the HF
    Trainer-based v3 pipeline in section 8 instead -- see the fairness note at the top."""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    start = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"{model_name} epoch {epoch + 1}/{epochs}", leave=False)
        for images, labels in pbar:
            x, labels = adapter(images).to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(-1) == labels).sum().item()
            total += labels.size(0)

        train_loss, train_acc = running_loss / total, correct / total
        val_acc, _, _, _ = evaluate_nn_model(model, adapter, val_loader, device)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(f"[{model_name}] epoch {epoch + 1}/{epochs} - loss {train_loss:.4f} "
              f"- train_acc {train_acc:.4f} - val_acc {val_acc:.4f}")

    train_time = time.time() - start
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else float("nan")
    return history, train_time, peak_mem_mb


def collect_model_metrics(model_for_eval, adapter, name, train_time_s, peak_mem_mb, save_path):
    """Runs the shared eval protocol (train/val/test accuracy, macro F1, latency,
    throughput, noise robustness, size) against an already-trained model, and returns
    a results dict in the same schema every model in this notebook reports into."""
    train_acc, _, _, _ = evaluate_nn_model(model_for_eval, adapter, train_loader, DEVICE)
    val_acc, val_f1, _, _ = evaluate_nn_model(model_for_eval, adapter, val_loader, DEVICE)
    test_acc, test_f1, y_true, y_pred = evaluate_nn_model(model_for_eval, adapter, test_loader, DEVICE)

    sample_batch = adapter(next(iter(test_loader))[0])
    latency_ms, throughput = measure_inference_speed(model_for_eval, sample_batch, DEVICE)
    size_mb = torch_model_size_mb(model_for_eval, save_path)
    noisy_acc = evaluate_with_noise(model_for_eval, adapter, test_loader, DEVICE, NOISE_STD)

    return {
        "model": name, "params": count_params(model_for_eval), "size_mb": size_mb,
        "train_acc": train_acc, "val_acc": val_acc, "test_acc": test_acc,
        "test_macro_f1": test_f1, "generalization_gap": train_acc - test_acc,
        "train_time_s": train_time_s, "peak_train_mem_mb": peak_mem_mb,
        "latency_ms": latency_ms, "throughput_sps": throughput,
        "noisy_test_acc": noisy_acc,
    }, (y_true, y_pred)


def collect_sklearn_model_metrics(model, name, save_path):
    """sklearn analog of collect_model_metrics above -- same schema, same eval
    protocol (train/val/test accuracy, macro F1, latency, throughput, noise
    robustness, size), just built on .fit()/.predict() instead of a forward
    pass. Assumes the model is already fit, and reuses the shared x_*_flat /
    y_*_np / noisy_x_test arrays section 9 builds once for every classical
    model below (same flattening Logistic Regression and Decision Tree
    already use -- pixel order doesn't matter to any of these).

    Doesn't return `params` or `train_time_s` -- what "parameter count" means
    differs per model (learned weights vs. node count vs. stored training
    points, see section 10's note), so each call site sets those two fields
    directly on the returned dict instead of this function guessing."""
    train_acc = accuracy_score(y_train_np, model.predict(x_train_flat))
    val_acc = accuracy_score(y_val_np, model.predict(x_val_flat))
    y_pred_test = model.predict(x_test_flat)
    test_acc = accuracy_score(y_test_np, y_pred_test)
    test_f1 = f1_score(y_test_np, y_pred_test, average="macro", zero_division=0)
    size_mb = sklearn_model_size_mb(model, save_path)

    single_times = []
    for _ in range(30):
        t0 = time.perf_counter()
        model.predict(x_test_flat[:1])
        single_times.append(time.perf_counter() - t0)
    latency_ms = float(np.mean(single_times) * 1000)

    t0 = time.perf_counter()
    for _ in range(10):
        model.predict(x_test_flat)
    throughput = 10 * len(x_test_flat) / (time.perf_counter() - t0)

    noisy_acc = accuracy_score(y_test_np, model.predict(noisy_x_test))

    return {
        "model": name, "size_mb": size_mb,
        "train_acc": train_acc, "val_acc": val_acc, "test_acc": test_acc,
        "test_macro_f1": test_f1, "generalization_gap": train_acc - test_acc,
        "peak_train_mem_mb": float("nan"),
        "latency_ms": latency_ms, "throughput_sps": float(throughput),
        "noisy_test_acc": noisy_acc,
    }, (y_test_np, y_pred_test)

## 7. Experiment Configuration

In [ ]:
BASELINE_EPOCHS = 15       # CNN, RNN, Transformer, Logistic Regression
LEARNING_RATE = 1e-3       # CNN, RNN, Transformer
NOISE_STD = 0.15           # shared robustness check, section 11

results = []
histories = {}
predictions = {}  # model_name -> (y_true, y_pred) on the test set, for confusion matrices

print(f"Baselines: {BASELINE_EPOCHS} epochs, batch size {BATCH_SIZE}, lr {LEARNING_RATE}")
print("S4D: its own v3 recipe, configured in section 8 below")

## 7.5 S4D Architecture Ablation: Validating Each Change

Before committing to the full recipe in section 8, this section adds the
roadmap's changes **one at a time** -- RGB, then patch embedding, then mean
pooling, then bigger capacity -- each with a quick `ABLATION_EPOCHS`-epoch
smoke test (plain Adam, no augmentation, no grouped optimizer -- section 8 is
where the real training recipe gets validated). The point isn't final accuracy,
it's whether each individual change is moving in the right direction before
spending a full training budget on the fully-assembled version.

These quick probes reuse `train_nn_model` from section 6 (the same harness
CNN/RNN/Transformer use) via `LogitsAdapter`, against the plain (non-augmented)
`train_loader`/`val_loader` from section 2 -- which are RGB, since `COLORED=True`
is already this notebook's global setting. There's no re-run of the *original*
grayscale architecture here for that reason (it would need a second, separately
loaded grayscale dataset just for one comparison point) -- section 13's write-up
has that number already (74.35% test_acc, full 400-epoch run), quoted below as
context, not as something these 4-epoch probes are meant to match.

In [ ]:
def s4d_ablation_probe(model, label, epochs=ABLATION_EPOCHS):
    """Quick, cheap sanity check for one incremental S4D architecture change.
    Not the real recipe -- no augmentation, no grouped optimizer, no warm
    restarts -- just enough epochs to see whether accuracy is trending in a
    reasonable direction."""
    history, train_time, _ = train_nn_model(
        LogitsAdapter(model), adapt_s4d, train_loader, val_loader,
        epochs=epochs, lr=1e-3, device=DEVICE, model_name=label
    )
    print(f"\n>>> {label}\n    val_acc after {epochs} epochs = {history['val_acc'][-1]:.4f}  "
          f"(train_acc = {history['train_acc'][-1]:.4f}, {train_time:.1f}s)\n")
    return history


print("Reference point, not re-run here: the pre-roadmap architecture (grayscale, "
      "pixel-level scan, take-last pooling, 2 layers, d_model=64) reached 74.35% "
      "test_acc over a full 400-epoch run -- see section 13. Different metric "
      "(test_acc, not val_acc) and a vastly different epoch count, so treat it as "
      "context, not a bar the 4-epoch steps below are trying to clear.")

ablation_results = {}

**Step 1: + RGB.** Same architecture as the pre-roadmap baseline otherwise --
pixel-level scan (`patch_size=1`), take-last pooling, 2 layers, `d_model=64` --
just fed the 3-channel input `COLORED=True` now provides.

In [ ]:
step1_model = GalaxyClassifierS4DFast(
    s4_state=64, d_model=64, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=2, patch_size=1, pooling="last"
)
h1 = s4d_ablation_probe(step1_model, "Step 1: + RGB (colored=True)")
ablation_results["Step 1: + RGB"] = h1["val_acc"][-1]

**Step 2: + patch embedding.** Groups pixels into `S4D_PATCH_SIZE x
S4D_PATCH_SIZE` patches (the modified `HilbertScan` from section 5.1), cutting
the sequence length from 4096 down to `(64/S4D_PATCH_SIZE)^2`. Still take-last
pooling, still 2 layers, still `d_model=64` -- isolating patch embedding's
effect on its own.

In [ ]:
step2_model = GalaxyClassifierS4DFast(
    s4_state=64, d_model=64, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=2, patch_size=S4D_PATCH_SIZE, pooling="last"
)
seq_len = step2_model.hilbert_scan.num_patches
h2 = s4d_ablation_probe(step2_model, f"Step 2: + patch embedding (patch_size={S4D_PATCH_SIZE}, {seq_len}-length sequence)")
ablation_results["Step 2: + patch embedding"] = h2["val_acc"][-1]

**Step 3: + mean pooling.** Replaces take-last with averaging across the whole
(now much shorter) patch sequence -- on top of step 2, everything else held fixed.

In [ ]:
step3_model = GalaxyClassifierS4DFast(
    s4_state=64, d_model=64, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=2, patch_size=S4D_PATCH_SIZE, pooling="mean"
)
h3 = s4d_ablation_probe(step3_model, "Step 3: + mean pooling (replaces take-last)")
ablation_results["Step 3: + mean pooling"] = h3["val_acc"][-1]

**Step 4: + bigger model.** `d_model`/`d_state` up to `S4D_STATE`, and a 3rd
S4 layer (`S4D_NUM_LAYERS`) -- this is the fully-assembled config section 8
trains for real. Grouped optimizer, augmentation, warm restarts, gradient
clipping, and warmup are training-*recipe* choices, not architecture ones, so
they aren't tested here -- section 8 is where those get validated, on the
architecture this step confirms is worth training properly.

In [ ]:
step4_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE, pooling=S4D_POOLING
)
h4 = s4d_ablation_probe(step4_model, f"Step 4: + bigger model (d_model=d_state={S4D_STATE}, {S4D_NUM_LAYERS} layers) -- full final config")
ablation_results["Step 4: full final config"] = h4["val_acc"][-1]

del step1_model, step2_model, step3_model, step4_model  # free GPU memory before section 8's real run

**Step 5: + residual + LayerNorm.** Same full final config as Step 4, but each
S4D block becomes a pre-norm residual block -- `h = h + S4D(LayerNorm(h))`
instead of the bare `h = S4D(h)` every step above uses. `GalaxyClassifierS4DFast`
has no normalization or residual connections anywhere by default -- an
intentional memory-efficiency choice (section 13's writeup) -- which is
unusual for a 3-layer stacked SSM; the S4/S4D papers' own reference blocks use
exactly this shape. Worth checking directly: the full 400-epoch production run
(section 16) plateaus hard after epoch ~300 (see that section's per-epoch
table once you've run it -- validation accuracy peaks at epoch 304 and then
oscillates without a further upward trend through epoch 400), which is
consistent with an optimization difficulty a deeper unnormalized stack can hit
even when it has the raw capacity to do better. This step checks whether
adding norm+residual back changes that.

In [ ]:
step5_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE, pooling=S4D_POOLING,
    use_norm=True, use_residual=True
)
h5 = s4d_ablation_probe(step5_model, "Step 5: + residual + LayerNorm (on top of full final config)")
ablation_results["Step 5: + residual + LayerNorm"] = h5["val_acc"][-1]


**Step 6: last-timestep pooling instead of mean (on top of Step 5).** Holds
norm+residual fixed from Step 5 and changes only the pooling choice, to
isolate that one variable on its own. `S4DConv` is a strictly causal
convolution -- each position's output depends only on that position and
earlier ones in the Hilbert-ordered sequence, never later ones (the FFT
convolution computes a length-`2L` result and keeps only the first `L`
outputs, which is exactly what makes it causal rather than a full circular
convolution). That means the *last* position is the only one that has
actually seen the whole sequence; every earlier position is a partial,
under-informed summary by construction. Mean pooling averages all of them
together -- a good default for bidirectional/non-causal encoders, but
diluting the signal here by construction, not just in principle: this
notebook's own original architecture used take-last for exactly this reason,
and mean pooling was introduced by the roadmap on the separate, also
reasonable argument that it gives every token a direct gradient path (section
7.5's original Step 3). Both arguments are legitimate in general -- this step
measures which one actually wins on this task instead of assuming either.

In [ ]:
step6_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE, pooling="last",
    use_norm=True, use_residual=True
)
h6 = s4d_ablation_probe(step6_model, "Step 6: + last-timestep pooling instead of mean (on top of Step 5)")
ablation_results["Step 6: last pooling (vs mean)"] = h6["val_acc"][-1]

del step5_model, step6_model  # free GPU memory before section 8's real run


In [ ]:
step7_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE, pooling="last",
    use_norm=False, use_residual=False
)
h7 = s4d_ablation_probe(step7_model, "Step 7: + last-timestep pooling instead of mean (on top of Step 5)")
ablation_results["Step 7: last pooling (vs mean)"] = h7["val_acc"][-1]

**Step 8: conv patch stem instead of a linear one (on top of Step 4/7's
config -- norm/residual left off, per Steps 4-7).** Everything else held at
section 1's current values (`S4D_PATCH_SIZE=2`, `S4D_POOLING="last"`). The
production confusion matrix (section 16, once you've run it) has Smooth
Cigar and Edge-on Disk confused with each other far more than any other
pair -- a `Linear` on a flattened raw patch can only ever see pixels inside
that one patch, never anything crossing a patch boundary, which is exactly
where a feature like a faint dust lane would sit. Shrinking `S4D_PATCH_SIZE`
(4->2 in section 1) gives the model more, smaller patches but doesn't change
that limitation -- each one is still a sealed box. This step swaps in
`ConvPatchStem` (ordinary 2D convs, same inductive bias the CNN baseline
already benefits from) to check whether that's actually the bottleneck
before spending a full run finding out.

In [ ]:
step8_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE, pooling=S4D_POOLING,
    use_norm=False, use_residual=False, patch_embed="conv"
)
h8 = s4d_ablation_probe(step8_model, "Step 8: conv patch stem (vs linear, Step 4/7)")
ablation_results["Step 8: conv patch stem"] = h8["val_acc"][-1]
del step8_model

print(f"\nStep 4/7 (linear patch embed): {ablation_results['Step 4: full final config']:.4f} / "
      f"{ablation_results['Step 7: last pooling (vs mean)']:.4f}")
print(f"Step 8 (conv patch embed):     {ablation_results['Step 8: conv patch stem']:.4f}")


In [ ]:
step9_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=4, pooling=S4D_POOLING,
    use_norm=False, use_residual=False, patch_embed="conv"
)
h9 = s4d_ablation_probe(step9_model, "Step 9: conv patch stem, patch_size=4 (vs Step 8's patch_size=2)")
ablation_results["Step 9: conv stem, patch_size=4"] = h9["val_acc"][-1]
del step9_model

print(f"\nStep 8 (conv stem, patch_size=2): {ablation_results['Step 8: conv patch stem']:.4f}")
print(f"Step 9 (conv stem, patch_size=4): {ablation_results['Step 9: conv stem, patch_size=4']:.4f}")

**Step 10: conv patch stem at patch_size=2, with dropout (the untested
combination).** Steps 8/9 compared patch_size 2 vs 4 for the conv stem, but
neither included dropout -- at the time, `S4D_DROPOUT` was still 0.0. Section
16's actual production run since then used patch_size=**4** with the conv
stem plus `S4D_DROPOUT=0.2`, and reached 86.10% test accuracy -- a real,
substantial improvement over the pre-conv-stem 80.35% run. But Step 8
(patch_size=2, conv, no dropout) scored higher than Step 9 (patch_size=4,
conv, no dropout) at this same quick-probe length -- 0.7469 vs 0.7388 -- so
the smaller patch size looked better within the conv-stem family on its own.
Production went with patch_size=4 anyway (not a mistake, necessarily -- see
the note where `S4D_PATCH_SIZE` is set in section 1 for the likely reason:
patch_size=2 means a 1024-token sequence instead of 256, on top of the conv
stem's own extra cost, which matters a lot at a 600+ epoch budget). This step
just checks what Step 8 would have shown with the regularization production
actually shipped with, before spending several more hours finding out the
hard way. If this clearly beats where Step 9 landed, patch_size=2 + conv +
dropout is worth a real run despite the extra compute; if it doesn't, the
current production config already found the better trade-off.

In [ ]:
step10_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=2, pooling=S4D_POOLING,
    use_norm=False, use_residual=False, dropout=S4D_DROPOUT, patch_embed="conv"
)
h10 = s4d_ablation_probe(step10_model, "Step 10: conv stem, patch_size=2, dropout={:.2f} (vs Step 9's patch_size=4)".format(S4D_DROPOUT))
ablation_results["Step 10: conv stem, patch_size=2, with dropout"] = h10["val_acc"][-1]
del step10_model

print(f"\nStep 9 (conv stem, patch_size=4, no dropout):      {ablation_results['Step 9: conv stem, patch_size=4']:.4f}")
print(f"Step 10 (conv stem, patch_size=2, dropout={S4D_DROPOUT}):  {ablation_results['Step 10: conv stem, patch_size=2, with dropout']:.4f}")
print("\nNeither number is directly comparable to production's 0.8610 test_acc -- both are "
      f"{ABLATION_EPOCHS}-epoch smoke tests on the plain-Adam ablation harness, not the full "
      "recipe. Read this as a relative signal (is patch_size=2 worth the extra compute), not "
      "an absolute prediction of the full run's outcome.")


### Ablation Summary

In [ ]:
print(f"S4D Architecture Ablation Summary (val_acc after {ABLATION_EPOCHS} quick epochs each)")
print("-" * 60)
for label, acc in ablation_results.items():
    print(f"{label:<45} {acc:>10.4f}")
print("-" * 60)
print("Each row should be flat-to-up from the one before it. A step that drops\n"
      "accuracy is worth a second look before section 8 spends BASELINE_EPOCHS on it.")

## 8. Train S4D (Comparison Run -- BASELINE_EPOCHS)

Section 7.5 validated each architecture change on its own; this trains the
fully-assembled version (`S4D_STATE`-dim, `S4D_NUM_LAYERS` layers,
`S4D_PATCH_SIZE`-patch embedding, `S4D_POOLING` pooling) for real --
`GalaxyDatasetAug` with rotation/flip augmentation, `build_grouped_optimizer`
(zero weight decay on `log_dt` / `log_A_real` / `A_imag`, the roadmap's
recommended weight decay elsewhere), a linear warmup into cosine annealing
with warm restarts, and explicit gradient clipping -- but capped at
`BASELINE_EPOCHS` (15) for this run, matching every other model in the
notebook, so the comparison below is a genuine apples-to-apples read on
accuracy rather than "S4D given N times the epochs." Checkpoints save every
epoch to `CHECKPOINT_DIR_COMPARISON` and resume automatically if this cell is
interrupted and re-run -- safe to stop early.

This is *not* S4D's best possible result -- section 16, at the end of the
notebook, retrains it from scratch at its full production epoch budget once
this comparison confirms it's worth doing.

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# Default: local, ephemeral (won't survive a Colab disconnect/reallocation)
CHECKPOINT_DIR = "./galaxy_s4_checkpoints"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = "/content/drive/MyDrive/GalaxyS4_Checkpoints"
    print(f"Google Drive mounted -- checkpoints will persist across disconnects at:\n  {CHECKPOINT_DIR}")
except ImportError:
    print("Not running in Colab -- checkpoints will only be saved locally, "
          f"at {CHECKPOINT_DIR}, which will NOT survive a runtime reset.")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# This comparison run (BASELINE_EPOCHS budget) and the full production run in
# section 16 (S4D_FINAL_EPOCHS budget, a different LR schedule) get separate
# checkpoint subdirectories -- their checkpoints aren't interchangeable, so
# they shouldn't share a resume path.
# New path -- the old CHECKPOINT_DIR_COMPARISON has checkpoints from the
# pre-residual architecture (fewer optimizer param groups: no LayerNorm/
# dropout params), which is why resume_from_checkpoint just raised a
# ValueError on param-group size. A fresh directory sidesteps that instead
# of trying to make an incompatible checkpoint load.
CHECKPOINT_DIR_COMPARISON = os.path.join(
    os.path.dirname(CHECKPOINT_DIR), f"GalaxyS4_Checkpoints_comparison_{s4d_config_tag}"
)
os.makedirs(CHECKPOINT_DIR_COMPARISON, exist_ok=True)

In [ ]:
class HF_GalaxyModelWrapper(nn.Module):
    """Makes GalaxyClassifierS4DFast compatible with the HF Trainer's dict-based I/O."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, inputs=None, labels=None, **kwargs):
        if inputs is None and 'inputs' in kwargs:
            inputs = kwargs['inputs']
        logits = self.model(inputs, return_logits=True)
        loss = None
        if labels is not None:
            if labels.ndim > 1 and labels.shape[-1] > 1:
                labels = torch.argmax(labels, dim=-1)
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if labels.ndim > 1 and labels.shape[-1] > 1:
        labels = np.argmax(labels, axis=-1)
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}


import torchvision.transforms.functional as TF

class GalaxyDatasetAug(Dataset):
    """Rotation/flip augmentation -- galaxies have no "up", so this is a free way to
    multiply the effective training set without introducing any labeling ambiguity.
    Arbitrary-angle rotation (was: random.choice of the four 90-degree multiples) --
    orientation is continuous, not 4-fold symmetric, so the discrete version was only
    ever showing the model 4 fixed views per image. Higher-value now that section 16.3
    shows real overfitting (train_acc 0.95 vs test_acc 0.83) the 6400-image set alone
    isn't preventing."""

    def __init__(self, x, y, augment=False):
        self.x, self.y, self.augment = x, y, augment

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        if self.augment:
            angle = random.uniform(0, 360)
            img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR, fill=0)
            if random.random() < 0.5:
                img = torch.flip(img, dims=(-1,))
        return {"inputs": img, "labels": self.y[idx]}


def build_grouped_optimizer(model, lr, weight_decay):
    """AdamW with weight decay excluded for the SSM core parameters (log_dt,
    log_A_real, A_imag) -- these set each channel's decay rate and oscillation
    frequency, and standard weight decay pulls in exactly the wrong direction on them."""
    ssm_core, other = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (ssm_core if any(k in name for k in ("log_dt", "log_A_real", "A_imag")) else other).append(p)
    return torch.optim.AdamW([
        {"params": ssm_core, "weight_decay": 0.0},
        {"params": other, "weight_decay": weight_decay},
    ], lr=lr)

In [ ]:
  torch.manual_seed(RNG_SEED)
fast_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE, pooling=S4D_POOLING,
    use_norm=S4D_USE_NORM, use_residual=S4D_USE_RESIDUAL, dropout=S4D_DROPOUT,
    patch_embed=S4D_PATCH_EMBED  # was missing -- this run was silently still using the
                                  # pre-conv-stem architecture, out of sync with section 16
).to(DEVICE)
wrapped_model = HF_GalaxyModelWrapper(fast_model)

train_dataset = GalaxyDatasetAug(x_train, y_train_onehot, augment=True)
eval_dataset = GalaxyDatasetAug(x_val, y_val_onehot, augment=False)

# Comparison-run budget: BASELINE_EPOCHS (15), same as every other model in this
# notebook -- see the fairness note at the top. S4D still keeps the training-
# recipe treatment it needs to train at all (weight decay excluded from the SSM
# core params, rotation/flip augmentation, warmup, gradient clipping) -- that
# was never what made the epoch mismatch unfair, only the epoch *count* itself
# is being matched now.
S4D_COMPARISON_EPOCHS = BASELINE_EPOCHS
s4d_steps_per_epoch = math.ceil(len(train_dataset) / S4D_BATCH_SIZE)

# lr=1e-3 and weight_decay=0.05 (on the non-SSM group) match the roadmap's
# recommended hyperparameters for this larger, patch-based architecture.
optimizer = build_grouped_optimizer(wrapped_model, lr=1e-3, weight_decay=0.05)

# Linear warmup (10% of total steps) into cosine annealing with warm restarts.
# T_0 scaled down from the full production recipe's T_0=10 epochs' worth of
# steps (see section 16): one restart roughly a third of the way through this
# much shorter budget, rather than a schedule tuned for a run 25x longer.
restart_epoch = max(1, S4D_COMPARISON_EPOCHS // 3)
warmup_steps = max(1, int(0.1 * S4D_COMPARISON_EPOCHS * s4d_steps_per_epoch))
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps)
restart_scheduler = CosineAnnealingWarmRestarts(
    optimizer, T_0=restart_epoch * s4d_steps_per_epoch, T_mult=2, eta_min=1e-5
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, restart_scheduler], milestones=[warmup_steps]
)

s4d_training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR_COMPARISON,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    logging_steps=50,
    learning_rate=1e-3,
    per_device_train_batch_size=S4D_BATCH_SIZE,
    per_device_eval_batch_size=S4D_BATCH_SIZE,
    num_train_epochs=S4D_COMPARISON_EPOCHS,   #changed to 2 for testing an error
    weight_decay=0.05,               # enforced per-group by build_grouped_optimizer above
    max_grad_norm=1.0,               # gradient clipping -- matters more at this larger capacity
    lr_scheduler_type="constant",    # HF's built-in scheduler is disabled; ours (warmup + warm restarts) drives LR
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    dataloader_num_workers=2,
    dataloader_persistent_workers=True,
    push_to_hub=False,
    report_to="none",
)

s4d_trainer = Trainer(
    model=wrapped_model,
    args=s4d_training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler),
)

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
s4d_start = time.time()

last_checkpoint = None  # comparison run is cheap -- not worth ever silently resuming
                          # a stale one again. (Section 16 still resumes -- that
                          # one actually costs real time to redo from scratch.)
s4d_trainer.train(resume_from_checkpoint=last_checkpoint)

s4d_train_time = time.time() - s4d_start
s4d_peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if DEVICE == "cuda" else float("nan")

# fast_model now holds the best checkpoint's weights (load_best_model_at_end=True)
fast_model.eval()
print(f"\nS4D comparison-run training complete in {s4d_train_time:.1f}s ({S4D_COMPARISON_EPOCHS} epochs)")

In [ ]:
# Pull the per-logging-step training curve out of the HF Trainer's log history,
# AND grab the fractional 'epoch' value HF logs alongside every entry so S4D can
# be overlaid directly on the epoch-based baseline plots in section 11 -- with
# eval_strategy="epoch" above, no step->epoch conversion math is needed.
s4d_log_epochs, s4d_log_loss = [], []
s4d_eval_epochs, s4d_eval_acc = [], []
for log in s4d_trainer.state.log_history:
    if "loss" in log:
        s4d_log_epochs.append(log["epoch"])
        s4d_log_loss.append(log["loss"])
    if "eval_accuracy" in log:
        s4d_eval_epochs.append(log["epoch"])
        s4d_eval_acc.append(log["eval_accuracy"])

histories["S4D"] = {"epochs": s4d_log_epochs, "loss": s4d_log_loss,
                     "eval_epochs": s4d_eval_epochs, "eval_acc": s4d_eval_acc}

In [ ]:
# Same evaluation protocol every other model gets (section 6) -- this is what makes
# the results table apples-to-apples despite the very different training procedure.
s4d_for_eval = LogitsAdapter(fast_model)
s4d_result, (s4d_y_true, s4d_y_pred) = collect_model_metrics(
    s4d_for_eval, adapt_s4d, "S4D", s4d_train_time, s4d_peak_mem_mb,
    "model_comparison_outputs/S4D.pt"
)
results.append(s4d_result)
predictions["S4D"] = (s4d_y_true, s4d_y_pred)

print(f"S4D: train_acc={s4d_result['train_acc']:.4f}  val_acc={s4d_result['val_acc']:.4f}  "
      f"test_acc={s4d_result['test_acc']:.4f}  macro_f1={s4d_result['test_macro_f1']:.4f}")

### 8.1 S4D-Specific: Test-Time Augmentation

The main training notebook also evaluates S4D with test-time augmentation --
averaging predictions across multiple rotations/flips of each test image. This
runs against the `BASELINE_EPOCHS`-budget comparison model from above, not the
full-budget production model -- section 16 repeats this same check against the
fully trained model. This is **not** applied to any of the other five models
in this notebook, so don't use it to directly compare rankings against them --
it's supplementary context on how much additional accuracy TTA buys
specifically for S4D, not part of the fair comparison.

In [ ]:
def predict_with_tta_and_scans(model, images, device, use_flips=True, batch_size=256):
    """Evaluates the model on multiple orientations of each input image and averages
    the resulting probabilities. Processes in batches to avoid OOM during the FFT ops."""
    model.eval()
    images = images.to(device)
    all_probs = []

    for i in range(0, len(images), batch_size):
        batch_imgs = images[i:i + batch_size]
        orientations = []
        for k in range(4):
            rot_img = torch.rot90(batch_imgs, k=k, dims=(-2, -1))
            orientations.append(rot_img)
            if use_flips:
                orientations.append(torch.flip(rot_img, dims=(-1,)))

        batch_probs = []
        with torch.no_grad():
            for oriented_img in orientations:
                # rot90/flip change memory strides; the Hilbert scan's .view() needs contiguous memory
                oriented_img = oriented_img.contiguous()
                logits = model(oriented_img, return_logits=True)
                batch_probs.append(torch.softmax(logits, dim=-1))

        all_probs.append(torch.stack(batch_probs).mean(dim=0))

    final_avg_probs = torch.cat(all_probs, dim=0)
    return final_avg_probs.argmax(dim=-1), final_avg_probs


preds_scans, _ = predict_with_tta_and_scans(fast_model, X_test, DEVICE, use_flips=False, batch_size=256)
acc_scans = (preds_scans.cpu() == y_test.cpu()).float().mean().item()

preds_tta, _ = predict_with_tta_and_scans(fast_model, X_test, DEVICE, use_flips=True, batch_size=256)
acc_tta = (preds_tta.cpu() == y_test.cpu()).float().mean().item()

print(f"S4D plain test accuracy:            {s4d_result['test_acc']:.4f}")
print(f"S4D multi-scan accuracy (4 orient.): {acc_scans:.4f}")
print(f"S4D TTA accuracy (8 orientations):   {acc_tta:.4f}")

## 9. Train the Baselines (CNN, RNN, Transformer, Logistic Regression, Decision Tree, SVM, KNN, Naive Bayes, Random Forest, Bayesian Logistic Regression)

CNN, RNN, Transformer, and Logistic Regression share one training budget
(`BASELINE_EPOCHS`, `LEARNING_RATE`, `BATCH_SIZE`, same seed) -- S4D above now
trains on the same `BASELINE_EPOCHS` budget too (with its own
architecture-appropriate optimizer and augmentation, section 8), so every
epoch-based model in this comparison shares the same 15-epoch budget. CNN,
RNN, and Transformer are also now capacity-matched to S4D: CNN's final hidden
layer and RNN's/Transformer's `d_model` all use `S4D_STATE` (section 1's
config), and all three take the same RGB input S4D does, via `IN_CHANNELS`.

The remaining six -- Decision Tree, SVM, KNN, Naive Bayes, Random Forest, and
Bayesian Logistic Regression -- have no epoch structure at all (section 5.5),
so there's no budget for them to share; each gets a single `.fit()` call
instead, evaluated through the same `collect_sklearn_model_metrics` harness
(section 6).

In [ ]:
# Capacity- and input-matched to S4D: same S4D_STATE for CNN's hidden width and
# RNN's/Transformer's d_model, same IN_CHANNELS (RGB or grayscale) as S4D sees.
# RNN/Transformer's own patch grid (8x8 patches, defined in section 3) is kept
# independent of S4D's -- different architectures, different sequence-length
# constraints -- only patch_dim's channel count needs to track COLORED.
BASELINE_PATCH_DIM = 64 * IN_CHANNELS   # 8x8-pixel patches (64 px) * channels

nn_model_specs = {
    "CNN": (GalaxyCNN(num_classes=NUM_CLASSES, in_channels=IN_CHANNELS, hidden_dim=S4D_STATE), adapt_cnn),
    "RNN (LSTM)": (GalaxyRNN(num_classes=NUM_CLASSES, patch_dim=BASELINE_PATCH_DIM, d_model=S4D_STATE), adapt_patches),
    "Transformer": (GalaxyTransformer(num_classes=NUM_CLASSES, patch_dim=BASELINE_PATCH_DIM,
                                       d_model=S4D_STATE, dim_ff=2 * S4D_STATE), adapt_patches),
}

for name, (model, adapter) in nn_model_specs.items():
    print(f"\n{'=' * 60}\nTraining {name}\n{'=' * 60}")
    history, train_time, peak_mem_mb = train_nn_model(
        model, adapter, train_loader, val_loader, BASELINE_EPOCHS, LEARNING_RATE, DEVICE, model_name=name
    )
    histories[name] = history
    result, (y_true, y_pred) = collect_model_metrics(
        model, adapter, name, train_time, peak_mem_mb,
        f"model_comparison_outputs/{name.replace(' ', '_')}.pt"
    )
    results.append(result)
    predictions[name] = (y_true, y_pred)

In [ ]:
# Redefine the function to use .reshape() instead of .view()
# to handle non-contiguous tensors created by train_test_split
def adapt_flat_numpy(images):
    """Logistic Regression and Decision Tree want a flat feature vector per sample.
    Pixel order doesn't matter to either (they treat each column independently),
    so a plain flatten is equivalent to a Hilbert-ordered one here, and cheaper."""
    return images.reshape(images.size(0), -1).numpy()

# --- Logistic Regression: SGD-trained, epoch-by-epoch, same BASELINE_EPOCHS budget ---
x_train_flat, x_val_flat, x_test_flat = adapt_flat_numpy(x_train), adapt_flat_numpy(x_val), adapt_flat_numpy(X_test)
y_train_np, y_val_np, y_test_np = y_train.numpy(), y_val.numpy(), y_test.numpy()

logreg = SGDClassifier(loss="log_loss", random_state=RNG_SEED)
classes = np.arange(NUM_CLASSES)
rng = np.random.RandomState(RNG_SEED)

logreg_history = {"train_acc": [], "val_acc": []}
start = time.time()
for epoch in range(BASELINE_EPOCHS):
    order = rng.permutation(len(x_train_flat))
    for i in range(0, len(order), BATCH_SIZE):
        idx = order[i:i + BATCH_SIZE]
        logreg.partial_fit(x_train_flat[idx], y_train_np[idx], classes=classes)
    train_acc = accuracy_score(y_train_np, logreg.predict(x_train_flat))
    val_acc = accuracy_score(y_val_np, logreg.predict(x_val_flat))
    logreg_history["train_acc"].append(train_acc)
    logreg_history["val_acc"].append(val_acc)
    print(f"[Logistic Regression] epoch {epoch + 1}/{BASELINE_EPOCHS} - train_acc {train_acc:.4f} - val_acc {val_acc:.4f}")
logreg_time = time.time() - start
histories["Logistic Regression"] = logreg_history

y_pred_test = logreg.predict(x_test_flat)
predictions["Logistic Regression"] = (y_test_np, y_pred_test)
test_acc = accuracy_score(y_test_np, y_pred_test)
test_f1 = f1_score(y_test_np, y_pred_test, average="macro", zero_division=0)
size_mb = sklearn_model_size_mb(logreg, "model_comparison_outputs/logreg.joblib")

single_times = []
for _ in range(30):
    t0 = time.perf_counter()
    logreg.predict(x_test_flat[:1])
    single_times.append(time.perf_counter() - t0)
latency_ms = float(np.mean(single_times) * 1000)
t0 = time.perf_counter()
for _ in range(10):
    logreg.predict(x_test_flat)
throughput = 10 * len(x_test_flat) / (time.perf_counter() - t0)

noisy_x_test = np.clip(x_test_flat + rng.normal(0, NOISE_STD, x_test_flat.shape), 0, 1)
noisy_acc = accuracy_score(y_test_np, logreg.predict(noisy_x_test))

results.append({
    "model": "Logistic Regression", "params": logreg.coef_.size + logreg.intercept_.size, "size_mb": size_mb,
    "train_acc": logreg_history["train_acc"][-1], "val_acc": logreg_history["val_acc"][-1], "test_acc": test_acc,
    "test_macro_f1": test_f1, "generalization_gap": logreg_history["train_acc"][-1] - test_acc,
    "train_time_s": logreg_time, "peak_train_mem_mb": float("nan"),
    "latency_ms": latency_ms, "throughput_sps": float(throughput),
    "noisy_test_acc": noisy_acc,
})

In [ ]:
# --- Decision Tree: single .fit() call, no epoch structure ---
tree = DecisionTreeClassifier(max_depth=16, random_state=RNG_SEED)

start = time.time()
tree.fit(x_train_flat, y_train_np)
tree_time = time.time() - start

train_acc = accuracy_score(y_train_np, tree.predict(x_train_flat))
val_acc = accuracy_score(y_val_np, tree.predict(x_val_flat))
y_pred_test = tree.predict(x_test_flat)
predictions["Decision Tree"] = (y_test_np, y_pred_test)
test_acc = accuracy_score(y_test_np, y_pred_test)
test_f1 = f1_score(y_test_np, y_pred_test, average="macro", zero_division=0)
size_mb = sklearn_model_size_mb(tree, "model_comparison_outputs/tree.joblib")

single_times = []
for _ in range(30):
    t0 = time.perf_counter()
    tree.predict(x_test_flat[:1])
    single_times.append(time.perf_counter() - t0)
latency_ms = float(np.mean(single_times) * 1000)
t0 = time.perf_counter()
for _ in range(10):
    tree.predict(x_test_flat)
throughput = 10 * len(x_test_flat) / (time.perf_counter() - t0)

noisy_acc = accuracy_score(y_test_np, tree.predict(noisy_x_test))

print(f"[Decision Tree] single fit - train_acc {train_acc:.4f} - val_acc {val_acc:.4f} "
      f"- {tree.tree_.node_count} nodes - depth {tree.get_depth()}")

results.append({
    "model": "Decision Tree", "params": tree.tree_.node_count, "size_mb": size_mb,  # node count, NOT weights
    "train_acc": train_acc, "val_acc": val_acc, "test_acc": test_acc,
    "test_macro_f1": test_f1, "generalization_gap": train_acc - test_acc,
    "train_time_s": tree_time, "peak_train_mem_mb": float("nan"),
    "latency_ms": latency_ms, "throughput_sps": float(throughput),
    "noisy_test_acc": noisy_acc,
})

In [ ]:
# --- SVM: single .fit() call, no epoch structure (like Decision Tree). RBF
# kernel -- the actual kernel trick, not just a linear boundary (Logistic
# Regression already covers that case). Slowest of the five new baselines to
# fit at this feature count -- a few minutes on a Colab CPU is normal.
svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RNG_SEED)

start = time.time()
svm_model.fit(x_train_flat, y_train_np)
svm_time = time.time() - start

svm_result, svm_preds = collect_sklearn_model_metrics(
    svm_model, "SVM (RBF)", "model_comparison_outputs/svm.joblib"
)
# What the RBF decision function actually needs stored: every support vector's
# coordinates, its dual coefficients, and the per-pair intercepts -- not a
# fixed weight count, scales with how many training points end up as support
# vectors.
svm_result["params"] = (svm_model.support_vectors_.size + svm_model.dual_coef_.size
                         + svm_model.intercept_.size)
svm_result["train_time_s"] = svm_time
predictions["SVM (RBF)"] = svm_preds
results.append(svm_result)
print(f"[SVM (RBF)] fit in {svm_time:.1f}s - train_acc {svm_result['train_acc']:.4f} "
      f"- val_acc {svm_result['val_acc']:.4f} "
      f"- {svm_model.support_vectors_.shape[0]} support vectors")

In [ ]:
# --- KNN: .fit() just stores the training set -- no real "training" happens
# until predict() runs the nearest-neighbor search. n_neighbors=5 is sklearn's
# default and a common textbook choice; not tuned further here, same spirit
# as Decision Tree's untuned max_depth=16.
knn_model = KNeighborsClassifier(n_neighbors=5)

start = time.time()
knn_model.fit(x_train_flat, y_train_np)
knn_time = time.time() - start

knn_result, knn_preds = collect_sklearn_model_metrics(
    knn_model, "KNN", "model_comparison_outputs/knn.joblib"
)
# KNN has no learned weights -- "fitting" just stores the training set, so the
# closest thing it has to a parameter count is how many numbers that storage
# takes: every training pixel value, once. Not comparable to a weight count.
knn_result["params"] = x_train_flat.size
knn_result["train_time_s"] = knn_time
predictions["KNN"] = knn_preds
results.append(knn_result)
print(f"[KNN] fit in {knn_time:.1f}s - train_acc {knn_result['train_acc']:.4f} "
      f"- val_acc {knn_result['val_acc']:.4f}")

In [ ]:
# --- Naive Bayes: closed-form, single .fit() call. GaussianNB, not
# MultinomialNB/BernoulliNB -- pixel intensities are continuous, not counts.
nb_model = GaussianNB()

start = time.time()
nb_model.fit(x_train_flat, y_train_np)
nb_time = time.time() - start

nb_result, nb_preds = collect_sklearn_model_metrics(
    nb_model, "Naive Bayes", "model_comparison_outputs/naive_bayes.joblib"
)
nb_result["params"] = nb_model.theta_.size + nb_model.var_.size + nb_model.class_prior_.size
nb_result["train_time_s"] = nb_time
predictions["Naive Bayes"] = nb_preds
results.append(nb_result)
print(f"[Naive Bayes] fit in {nb_time:.1f}s - train_acc {nb_result['train_acc']:.4f} "
      f"- val_acc {nb_result['val_acc']:.4f}")

In [ ]:
# --- Random Forest: single .fit() call, same story as Decision Tree but
# bagged over n_estimators trees. sklearn's default n_estimators=100;
# max_depth left unconstrained -- bagging + feature subsampling regularizes
# an ensemble of deep trees in a way a single deep tree doesn't get for free,
# so this doesn't need Decision Tree's max_depth=16 cap to behave.
rf_model = RandomForestClassifier(n_estimators=100, random_state=RNG_SEED, n_jobs=-1)

start = time.time()
rf_model.fit(x_train_flat, y_train_np)
rf_time = time.time() - start

rf_result, rf_preds = collect_sklearn_model_metrics(
    rf_model, "Random Forest", "model_comparison_outputs/random_forest.joblib"
)
# Same convention as the single Decision Tree above -- summed node count
# across all n_estimators trees, not learned weights.
rf_total_nodes = sum(t.tree_.node_count for t in rf_model.estimators_)
rf_result["params"] = rf_total_nodes
rf_result["train_time_s"] = rf_time
predictions["Random Forest"] = rf_preds
results.append(rf_result)
print(f"[Random Forest] fit in {rf_time:.1f}s - train_acc {rf_result['train_acc']:.4f} "
      f"- val_acc {rf_result['val_acc']:.4f} - {rf_total_nodes:,} total nodes "
      f"across {len(rf_model.estimators_)} trees")

In [ ]:
# --- Bayesian Logistic Regression (section 5.6): single .fit() call --
# LogisticRegression's own lbfgs solver iterates internally for the MAP fit,
# no outer epoch loop exposed here, same as every other classical model above.
blr_model = BayesianLogisticRegression(C=1.0, max_iter=200, random_state=RNG_SEED)

start = time.time()
blr_model.fit(x_train_flat, y_train_np)
blr_time = time.time() - start

blr_result, blr_preds = collect_sklearn_model_metrics(
    blr_model, "Bayesian Logistic Regression", "model_comparison_outputs/bayesian_logreg.joblib"
)
# Same convention as plain Logistic Regression -- the MAP weights are the
# parameters; the per-weight posterior variances describe uncertainty about
# those same parameters rather than adding new ones (size_mb above does
# reflect the extra storage for them, params doesn't).
blr_result["params"] = blr_model.coef_.size + blr_model.intercept_.size
blr_result["train_time_s"] = blr_time
predictions["Bayesian Logistic Regression"] = blr_preds
results.append(blr_result)
print(f"[Bayesian Logistic Regression] fit in {blr_time:.1f}s - train_acc {blr_result['train_acc']:.4f} "
      f"- val_acc {blr_result['val_acc']:.4f}")

print("\nAll eleven models trained and evaluated.")

### 9.5 Two-Stage (Cascade) Classifier -- Comparison Budget

Every full production run so far -- four of them now, spanning mean pooling,
last-timestep pooling, the conv patch stem, and dropout+wider augmentation --
has left the Smooth Cigar / Edge-on Disk confusion essentially untouched
(~200-230 errors each time, **68.7%** of all remaining errors in the current
86.1% production model). A single 4-way classifier has to split its capacity
four ways; this tries giving the one genuinely hard distinction a model that
only ever has to make *that* decision instead.

**Design:** a router (`GalaxyClassifierS4DFast`, `num_classes=2`) splits
"elongated" (Smooth Cigar + Edge-on Disk) from "round-ish" (Smooth Round +
Unbarred Spiral) -- a distinction the current production model already gets
right the large majority of the time, per its own confusion matrix. Whichever
branch the router picks, a second, dedicated binary specialist resolves the
final class: Cigar-vs-Disk if elongated, Round-vs-Spiral if not. All three
stages share the exact same backbone config as the current production S4D
(`S4D_PATCH_EMBED`, `S4D_PATCH_SIZE`, `S4D_POOLING`, `S4D_USE_NORM`,
`S4D_USE_RESIDUAL`, `S4D_DROPOUT`) -- the only variable under test is whether
splitting the problem into a cascade helps, not a simultaneously-different
backbone too.

**This does not touch or replace `final_model` / section 16 in any way.** All
three stages here train at the shared `BASELINE_EPOCHS` budget, same as every
other row in section 10's table below -- exactly the same "cheap comparison
check before a real commitment" role S4D's own section 8 entry plays relative
to its section 16 production run. If this looks promising here, the natural
next step is a full-budget version, the same way S4D's own comparison run
earned its production run.

Two honest caveats baked into how this gets scored below:
- `params` / `size_mb` are the **sum of all three stages** -- the real cost of
  deploying this instead of one model, not a discount for "each one is smaller."
- `latency_ms` / `throughput_sps` run all three stages unconditionally for
  every sample, including whichever specialist branch gets thrown away. A
  real deployment would branch and only run the router plus the one specialist
  that applies -- so these two numbers are a **pessimistic upper bound**, not
  what a deployed version would actually cost.

In [ ]:
# Class indices, per CLASS_NAMES: 0=Smooth Round, 1=Smooth Cigar,
# 2=Edge-on Disk, 3=Unbarred Spiral.
ELONGATED_CLASSES = (1, 2)   # Smooth Cigar, Edge-on Disk -- the hard pair
ROUND_CLASSES = (0, 3)       # Smooth Round, Unbarred Spiral -- already easy

def to_router_labels(y):
    """1 = elongated (Cigar/Disk), 0 = round-ish (Round/Spiral)."""
    return torch.isin(y, torch.tensor(ELONGATED_CLASSES)).long()

def filter_and_remap(x, y, classes):
    """Keeps only the two given original classes and remaps them to {0, 1}
    in the order given -- classes[0] -> 0, classes[1] -> 1."""
    mask = torch.isin(y, torch.tensor(classes))
    y_sub = y[mask]
    y_remapped = torch.where(y_sub == classes[0], 0, 1)
    return x[mask], y_remapped

# Router: full train/val set, relabeled as elongated-vs-round-ish.
y_train_router = to_router_labels(y_train)
y_val_router = to_router_labels(y_val)

# Elongated specialist: Cigar/Disk images only, remapped 1->0 (Cigar), 2->1 (Disk).
x_train_elong, y_train_elong = filter_and_remap(x_train, y_train, ELONGATED_CLASSES)
x_val_elong, y_val_elong = filter_and_remap(x_val, y_val, ELONGATED_CLASSES)

# Round-ish specialist: Round/Spiral images only, remapped 0->0 (Round), 3->1 (Spiral).
x_train_round, y_train_round = filter_and_remap(x_train, y_train, ROUND_CLASSES)
x_val_round, y_val_round = filter_and_remap(x_val, y_val, ROUND_CLASSES)

print(f"Router:               {len(x_train):5d} train / {len(x_val):4d} val (all classes)")
print(f"Elongated specialist: {len(x_train_elong):5d} train / {len(x_val_elong):4d} val "
      f"(Cigar={int((y_train_elong==0).sum())}, Disk={int((y_train_elong==1).sum())})")
print(f"Round-ish specialist: {len(x_train_round):5d} train / {len(x_val_round):4d} val "
      f"(Round={int((y_train_round==0).sum())}, Spiral={int((y_train_round==1).sum())})")

In [ ]:
CHECKPOINT_DIR_CASCADE_ROUTER = os.path.join(
    os.path.dirname(CHECKPOINT_DIR), f"GalaxyS4_Checkpoints_cascade_router_{s4d_config_tag}"
)
CHECKPOINT_DIR_CASCADE_ELONGATED = os.path.join(
    os.path.dirname(CHECKPOINT_DIR), f"GalaxyS4_Checkpoints_cascade_elongated_{s4d_config_tag}"
)
CHECKPOINT_DIR_CASCADE_ROUND = os.path.join(
    os.path.dirname(CHECKPOINT_DIR), f"GalaxyS4_Checkpoints_cascade_round_{s4d_config_tag}"
)
for _d in (CHECKPOINT_DIR_CASCADE_ROUTER, CHECKPOINT_DIR_CASCADE_ELONGATED, CHECKPOINT_DIR_CASCADE_ROUND):
    os.makedirs(_d, exist_ok=True)


def train_cascade_stage(x_train_sub, y_train_sub, x_val_sub, y_val_sub, stage_name, checkpoint_dir):
    """One binary GalaxyClassifierS4DFast stage for the cascade below -- same
    backbone config as production, same HF Trainer recipe as section 8's S4D
    comparison run (warmup + warm restarts, BASELINE_EPOCHS budget), just
    num_classes=2 and a filtered/relabeled slice of the data instead of all
    four classes at once. Always trains fresh (no resume) -- same reasoning
    as section 8's own S4D comparison run: cheap enough that resuming was
    never worth the risk of silently reusing a stale checkpoint."""
    torch.manual_seed(RNG_SEED)
    model = GalaxyClassifierS4DFast(
        s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=2, colored=COLORED,
        num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE, pooling=S4D_POOLING,
        use_norm=S4D_USE_NORM, use_residual=S4D_USE_RESIDUAL, dropout=S4D_DROPOUT,
        patch_embed=S4D_PATCH_EMBED
    ).to(DEVICE)
    wrapped = HF_GalaxyModelWrapper(model)

    stage_train_dataset = GalaxyDatasetAug(x_train_sub, y_train_sub, augment=True)
    stage_eval_dataset = GalaxyDatasetAug(x_val_sub, y_val_sub, augment=False)

    steps_per_epoch = math.ceil(len(stage_train_dataset) / S4D_BATCH_SIZE)
    optimizer = build_grouped_optimizer(wrapped, lr=1e-3, weight_decay=0.05)
    restart_epoch = max(1, BASELINE_EPOCHS // 3)
    warmup_steps = max(1, int(0.1 * BASELINE_EPOCHS * steps_per_epoch))
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps)
    restart_scheduler = CosineAnnealingWarmRestarts(
        optimizer, T_0=restart_epoch * steps_per_epoch, T_mult=2, eta_min=1e-5
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, restart_scheduler], milestones=[warmup_steps]
    )

    training_args = TrainingArguments(
        output_dir=checkpoint_dir,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=3,
        logging_steps=50, learning_rate=1e-3,
        per_device_train_batch_size=S4D_BATCH_SIZE, per_device_eval_batch_size=S4D_BATCH_SIZE,
        num_train_epochs=BASELINE_EPOCHS, weight_decay=0.05, max_grad_norm=1.0,
        lr_scheduler_type="constant", load_best_model_at_end=True,
        metric_for_best_model="accuracy", greater_is_better=True,
        dataloader_num_workers=2, dataloader_persistent_workers=True,
        push_to_hub=False, report_to="none",
    )
    trainer = Trainer(
        model=wrapped, args=training_args,
        train_dataset=stage_train_dataset, eval_dataset=stage_eval_dataset,
        compute_metrics=compute_metrics, optimizers=(optimizer, scheduler),
    )
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    start = time.time()
    trainer.train(resume_from_checkpoint=None)
    train_time = time.time() - start
    peak_mem = torch.cuda.max_memory_allocated() / 1e6 if DEVICE == "cuda" else float("nan")
    model.eval()
    print(f"[{stage_name}] trained in {train_time:.1f}s ({BASELINE_EPOCHS} epochs)")
    return model, train_time, peak_mem

In [ ]:
router_model, router_train_time, router_peak_mem = train_cascade_stage(
    x_train, y_train_router, x_val, y_val_router,
    "Router (elongated vs. round-ish)", CHECKPOINT_DIR_CASCADE_ROUTER
)

In [ ]:
elongated_model, elongated_train_time, elongated_peak_mem = train_cascade_stage(
    x_train_elong, y_train_elong, x_val_elong, y_val_elong,
    "Elongated specialist (Cigar vs. Disk)", CHECKPOINT_DIR_CASCADE_ELONGATED
)

In [ ]:
round_model, round_train_time, round_peak_mem = train_cascade_stage(
    x_train_round, y_train_round, x_val_round, y_val_round,
    "Round-ish specialist (Round vs. Spiral)", CHECKPOINT_DIR_CASCADE_ROUND
)

In [ ]:
class TwoStageCascade(nn.Module):
    """Combines the three stages above into a single (B, 4) logit-shaped
    output -- a drop-in "model" for collect_model_metrics and everything else
    in section 6's harness, even though it's really three separately-trained
    models wired together only at inference time. Whichever branch the router
    didn't pick gets -1e4 in its two class slots, so argmax always recovers
    exactly what the cascade actually decided -- no blending between branches."""

    def __init__(self, router, elongated_specialist, round_specialist):
        super().__init__()
        self.router = router
        self.elongated_specialist = elongated_specialist
        self.round_specialist = round_specialist

    def forward(self, x):
        router_logits = self.router(x, return_logits=True)                  # (B,2): [round-ish, elongated]
        elongated_logits = self.elongated_specialist(x, return_logits=True)  # (B,2): [Cigar, Disk]
        round_logits = self.round_specialist(x, return_logits=True)          # (B,2): [Round, Spiral]

        is_elongated = router_logits.argmax(dim=-1).bool()

        out = torch.full((x.size(0), NUM_CLASSES), -1e4, device=x.device, dtype=elongated_logits.dtype)
        out[is_elongated, 1] = elongated_logits[is_elongated, 0]   # Smooth Cigar
        out[is_elongated, 2] = elongated_logits[is_elongated, 1]   # Edge-on Disk
        out[~is_elongated, 0] = round_logits[~is_elongated, 0]     # Smooth Round
        out[~is_elongated, 3] = round_logits[~is_elongated, 1]     # Unbarred Spiral
        return out


cascade_model = TwoStageCascade(router_model, elongated_model, round_model).to(DEVICE)
cascade_model.eval()

cascade_train_time = router_train_time + elongated_train_time + round_train_time
cascade_peak_mem = (max(router_peak_mem, elongated_peak_mem, round_peak_mem)
                     if DEVICE == "cuda" else float("nan"))

cascade_result, cascade_preds = collect_model_metrics(
    cascade_model, adapt_s4d, "Two-Stage Classifier (comparison budget)",
    cascade_train_time, cascade_peak_mem,
    "model_comparison_outputs/two_stage_cascade.pt"
)
predictions["Two-Stage Classifier (comparison budget)"] = cascade_preds
results.append(cascade_result)

s4d_comparison_result = next(r for r in results if r["model"] == "S4D")
print(f"\nTwo-Stage Classifier (comparison budget): test_acc = {cascade_result['test_acc']:.4f}  "
      f"macro_f1 = {cascade_result['test_macro_f1']:.4f}")
print(f"S4D (comparison budget, same {BASELINE_EPOCHS}-epoch fairness bar):     "
      f"test_acc = {s4d_comparison_result['test_acc']:.4f}  "
      f"macro_f1 = {s4d_comparison_result['test_macro_f1']:.4f}")
print(f"Combined params: {cascade_result['params']:,} (router + both specialists) "
      f"vs. S4D's single-model {s4d_comparison_result['params']:,}")

## 10. Results Table

In [ ]:
results_df = pd.DataFrame(results).set_index("model")
results_df = results_df.round(4)
results_df.to_csv("model_comparison_outputs/results_table.csv")
results_df

`params` means something different across these eleven rows, same idea as the
Decision Tree note already made, just spelled out for everyone now:

- **Learned weights** (directly comparable to each other): S4D, CNN, RNN, Transformer,
  Logistic Regression, Bayesian Logistic Regression.
- **Node count, not weights**: Decision Tree, Random Forest (summed across all
  `n_estimators` trees).
- **Support vectors + dual coefficients, not weights in the usual sense**: SVM --
  counts what the RBF decision function needs stored, which scales with how many
  training points end up as support vectors, not with a fixed architecture.
- **Not a trained parameter count at all**: KNN has no learned weights -- `.fit()`
  just stores the training set, so its `params` here is every stored pixel value
  (`x_train_flat.size`), reported for comparability but not meaningful as "model
  complexity" the way it is for the other rows.

`peak_train_mem_mb` is only populated on a CUDA GPU, and only for the four PyTorch
models (S4D, CNN, RNN, Transformer) -- `NaN` for everything sklearn-based, including
the new five, since none of them report GPU memory. `train_time_s` is directly
comparable across CNN/RNN/Transformer/Logistic Regression/S4D (same `BASELINE_EPOCHS`
budget) but not for the six single-`.fit()` models, which don't have an epoch budget
to be matched on in the first place. S4D's row here is the `BASELINE_EPOCHS`-budget
comparison run (section 8) -- see section 16 for the full-epoch production run's
numbers, which aren't folded into this table since they're not on a matched epoch
budget with the rest. All four neural models (CNN, RNN, Transformer, S4D) are now
capacity-matched at `S4D_STATE` -- expect noticeably higher `params` and `size_mb`
for all of them than in a `S4D_STATE=64` run.

## 11. Visual Comparisons

In [ ]:
# S4D's own training curve -- logged at a finer grain (per logging_steps) than
# the epoch-level baselines, so it gets its own detailed view here even though
# it's on the same BASELINE_EPOCHS budget and also appears overlaid on the
# shared plot right below.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(histories["S4D"]["epochs"], histories["S4D"]["loss"])
axes[0].set_title(f"S4D Training Loss (per logging step, {S4D_COMPARISON_EPOCHS} epochs)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[1].plot(histories["S4D"]["eval_epochs"], histories["S4D"]["eval_acc"], color="seagreen", marker="o")
axes[1].set_title("S4D Validation Accuracy (per epoch)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
baseline_names = ["CNN", "RNN (LSTM)", "Transformer", "Logistic Regression"]
for name in baseline_names:
    hist = histories[name]
    axes[0].plot(hist["train_acc"], label=name, marker="o", markersize=3)
    axes[1].plot(hist["val_acc"], label=name, marker="o", markersize=3)

# S4D joins the validation-accuracy plot directly -- eval_strategy="epoch" in
# section 8 means histories["S4D"]["eval_epochs"] already lines up with the same
# 1..BASELINE_EPOCHS x-axis the baselines use, no step->epoch conversion needed.
axes[1].plot(histories["S4D"]["eval_epochs"], histories["S4D"]["eval_acc"],
             label="S4D", marker="s", markersize=5, color="crimson", linestyle="--")

# S4D never logs a "train_acc" metric (the HF Trainer only tracks training loss
# by default) -- rather than leave it out of the left plot entirely, its training
# loss goes on a secondary axis so both curves are visible together.
ax0_twin = axes[0].twinx()
ax0_twin.plot(histories["S4D"]["epochs"], histories["S4D"]["loss"],
              label="S4D (train loss, right axis)", color="crimson", linestyle=":", alpha=0.8)
ax0_twin.set_ylabel("S4D train loss", color="crimson")
ax0_twin.tick_params(axis="y", labelcolor="crimson")

axes[0].set_title("Training Accuracy by Epoch (S4D train loss on right axis)")
axes[1].set_title("Validation Accuracy by Epoch (S4D included)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")

lines_0, labels_0 = axes[0].get_legend_handles_labels()
lines_twin, labels_twin = ax0_twin.get_legend_handles_labels()
axes[0].legend(lines_0 + lines_twin, labels_0 + labels_twin, fontsize=8)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# S4D gets its own color across every panel so it's easy to track as the model
# this notebook is built to justify, against the five baselines in their
# original colors.
def bar_colors(series, highlight="crimson", base="steelblue"):
    return [highlight if idx == "S4D" else base for idx in series.index]

s = results_df["test_acc"].sort_values()
s.plot.barh(ax=axes[0, 0], color=bar_colors(s, "crimson", "steelblue"))
axes[0, 0].set_title("Test Accuracy")

s = results_df["test_macro_f1"].sort_values()
s.plot.barh(ax=axes[0, 1], color=bar_colors(s, "crimson", "seagreen"))
axes[0, 1].set_title("Test Macro F1")

s = results_df["params"].sort_values()
s.plot.barh(ax=axes[0, 2], color=bar_colors(s, "crimson", "indianred"), logx=True)
axes[0, 2].set_title("Parameter / Node Count (log scale)")

s = results_df["size_mb"].sort_values()
s.plot.barh(ax=axes[1, 0], color=bar_colors(s, "crimson", "goldenrod"))
axes[1, 0].set_title("Model Size on Disk (MB)")

s = results_df["latency_ms"].sort_values()
s.plot.barh(ax=axes[1, 1], color=bar_colors(s, "crimson", "mediumpurple"))
axes[1, 1].set_title("Single-Sample Inference Latency (ms)")

s = results_df["train_time_s"].sort_values()
s.plot.barh(ax=axes[1, 2], color=bar_colors(s, "crimson", "slategray"), logx=True)
axes[1, 2].set_title("Total Training Time (s, log scale)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(results_df["params"], results_df["test_acc"], s=80)
for name, row in results_df.iterrows():
    axes[0].annotate(name, (row["params"], row["test_acc"]), fontsize=8, xytext=(5, 5), textcoords="offset points")
axes[0].set_xscale("log")
axes[0].set_xlabel("Parameters / nodes (log scale)")
axes[0].set_ylabel("Test accuracy")
axes[0].set_title("Accuracy vs. Model Size")

axes[1].scatter(results_df["latency_ms"], results_df["test_acc"], s=80)
for name, row in results_df.iterrows():
    axes[1].annotate(name, (row["latency_ms"], row["test_acc"]), fontsize=8, xytext=(5, 5), textcoords="offset points")
axes[1].set_xlabel("Inference latency (ms/sample)")
axes[1].set_ylabel("Test accuracy")
axes[1].set_title("Accuracy vs. Inference Latency")

plt.tight_layout()
plt.show()

In [ ]:
# Sized off len(predictions) rather than a hardcoded 2x3 -- that grid only ever
# had exactly 6 slots for exactly 6 models; with eleven now in `predictions`,
# a fixed 2x3 would silently truncate to the first 6 via zip() and never plot
# the rest, no error either way. 4 columns keeps each heatmap a readable size
# regardless of how many models end up in the comparison.
n_cols = 4
n_rows = math.ceil(len(predictions) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes_flat = np.atleast_1d(axes).flatten()

for ax, (name, (y_true, y_pred)) in zip(axes_flat, predictions.items()):
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
    ax.set_title(f"{name} (test acc: {results_df.loc[name, 'test_acc']:.2%})", fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

# Turn off any leftover empty axes (grid slots > number of models).
for ax in axes_flat[len(predictions):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 12. Robustness Check

Accuracy on the same test set with additive Gaussian pixel noise -- a cheap proxy
for how much a model leans on brittle high-frequency detail versus overall shape.

In [ ]:
robustness_df = results_df[["test_acc", "noisy_test_acc"]].copy()
robustness_df["accuracy_drop"] = robustness_df["test_acc"] - robustness_df["noisy_test_acc"]
robustness_df.sort_values("accuracy_drop").plot.barh(y=["test_acc", "noisy_test_acc"], figsize=(9, 5))
plt.title(f"Clean vs. Noisy Test Accuracy (Gaussian noise, std={NOISE_STD})")
plt.xlabel("Accuracy")
plt.tight_layout()
plt.show()
robustness_df

## 13. Writing Up the Justification

> **Note:** the table and narrative below are from an old run of a since-
> superseded architecture -- grayscale, pixel-level Hilbert scan, take-last
> pooling, 2 layers, `d_model=64` -- trained for 400 epochs against 15-epoch
> baselines at their original (unmatched) capacity. Section 7.5's roadmap
> changes (RGB, patch embedding, mean pooling, `S4D_STATE`-dim capacity,
> capacity-matched baselines) mean every number below -- params, size, time,
> accuracy, all of it -- is from a different model than the one this notebook
> now trains. This section hasn't been re-run since; treat the table and the
> specific figures quoted in the prose as historical reference for *how to
> structure this kind of write-up*, not as current results. Re-run the
> notebook and rewrite this section once the new numbers are in.

**Results from the v3 pipeline (S4D: 400 epochs, grouped optimizer, augmentation, warm restarts; baselines: 15 epochs, shared generic recipe):**

| Model | Test Acc | Test Macro F1 | Params | Size (MB) | Gen. Gap | Train Time | Latency (ms) | Peak Train Mem (MB) | Noisy Test Acc |
|---|---|---|---|---|---|---|---|---|---|
| CNN | 80.75% | 0.805 | 285,988 | 1.15 | 10.39 pp | 13.7s | 0.56 | 146.6 | 25.4% |
| S4D | 74.35% | 0.744 | 17,028 | 0.11 | 1.70 pp | 3760.3s | 2.99 | 800.9 | 50.5% |
| Transformer | 74.20% | 0.742 | 75,716 | 0.31 | 3.53 pp | 15.4s | 0.90 | 60.4 | 55.6% |
| RNN (LSTM) | 67.35% | 0.677 | 70,980 | 0.29 | 2.82 pp | 9.8s | 0.71 | 83.9 | 53.5% |
| Decision Tree | 58.55% | 0.586 | 1,311* | 0.13 | 39.57 pp | 28.4s | 0.17 | – | 30.1% |
| Logistic Regression | 49.30% | 0.447 | 16,388 | 0.07 | 12.47 pp | 13.3s | 0.46 | – | 36.9% |

*node count, not learned weights.

### Does S4D reach competitive accuracy?

Yes — this is the headline change from the earlier run. S4D reaches 74.35% test accuracy and a macro F1 of 0.744, essentially tied with Transformer (74.20% / 0.742, a 0.15-point gap that's within run-to-run noise for a single seed) and 6.4 points behind CNN. This directly confirms the earlier diagnosis: S4D wasn't incapable of the task, it needed the SSM-aware optimizer treatment (grouped weight decay, warm restarts) that the earlier uniform-Adam run didn't give it.

### What does S4D cost, relative to what it buys?

This is now a real cost/benefit question, not a moot one:

- **Parameters/size**: 17,028 params, 0.11 MB — 16.8x fewer parameters and 10.9x smaller on disk than CNN, for a 6.4-point accuracy cost. Against Transformer specifically: 4.4x fewer parameters and ~3x smaller on disk, for *no* accuracy cost (S4D matches it).
- **Generalization gap**: 1.70 points, the tightest of all six models — tighter than Transformer (3.53) and RNN (2.82), far tighter than CNN (10.39) or the Decision Tree (39.57). One caveat below on how much to credit this to the architecture itself.
- **Training time**: 3760.3s (~62.7 min) vs. Transformer's 15.4s — about 244x longer. This isn't a fair per-epoch comparison by design (400 epochs vs. 15, see the fairness note at the top of the notebook), but it's a real cost if you're the one waiting on it.
- **Peak training memory**: 800.9 MB — 5.5x more than CNN, 13.3x more than Transformer. Notably *less* than the earlier run's simplified S4D (1858.4 MB) despite training far longer, because the real `S4DConv` architecture is smaller (uses `d_state // 2` complex modes internally, no extra LayerNorm/projection/dropout layers).
- **Inference latency**: 2.99 ms/sample, still the slowest of the six, but roughly half the earlier run's 5.88 ms — consistent with the smaller real architecture. Still 3.3x slower than Transformer and 5.3x slower than CNN, largely because the FFT convolution kernel gets rebuilt from scratch on every forward call rather than cached after training.

### Where do the alternatives fall short?

- **Decision Tree**: a 39.57-point train/test gap — by far the worst — confirms it's memorizing training pixels. Noisy accuracy (30.1%) collapses toward chance.
- **Logistic Regression**: 49.3% test accuracy is the "how hard is this with a linear boundary on raw pixels" floor — above chance, well below every model with spatial or sequential structure.
- **RNN (LSTM)**: 67.35% test accuracy is clearly behind the top three, though it has the best *relative* noise robustness of the six (79.4% of clean accuracy retained) and a tight generalization gap (2.82 points).
- **CNN**: highest clean accuracy (80.75%) but a striking weakness once noise enters the picture — accuracy collapses to 25.35%, the *worst* of all six models, worse than the Decision Tree. A model that looks best on the clean test set is the least trustworthy under any real-world sensor noise, which the S4D/RNN/Transformer family doesn't share nearly as badly.
- **Transformer**: matches S4D on clean accuracy with 4.4x more parameters, but gets there with a completely standard 15-epoch recipe and no architecture-specific optimizer tricks — the most direct "was the extra effort worth it" comparison, addressed below.

### What did the v3 fix actually change?

Comparing this run against the earlier naive-Adam run directly:

| | Naive Adam (earlier run) | v3 pipeline (this run) |
|---|---|---|
| Test accuracy | 24.0% | 74.35% (+50.35 pp) |
| Test macro F1 | 0.097 | 0.744 |
| Params | 42,116 | 17,028 |
| Peak train memory | 1858.4 MB | 800.9 MB |
| Latency | 5.88 ms | 2.99 ms |

A +50-point swing is about as clean a confirmation as this kind of comparison gets. One honest caveat: two things changed at once between these runs — the training recipe *and* the model itself (the earlier run used a simplified from-scratch reimplementation; this one uses the real, smaller `GalaxyClassifierS4DFast`). This isn't a controlled ablation that isolates which single ingredient mattered most, so "the grouped optimizer alone fixed it" is a stronger claim than the data supports — what the data does support is that the full v3 recipe (real architecture + zero weight decay on the SSM core params + warm restarts + augmentation + a much longer schedule) fixes what the naive recipe couldn't.

### Two caveats worth stating plainly

- **The augmentation confound**: S4D is the only model in this comparison trained with rotation/flip augmentation (`GalaxyDatasetAug`). Its unusually tight generalization gap (1.70 pp) is genuinely impressive, but some of that credit likely belongs to augmentation rather than the architecture alone — none of the other five models got the same benefit, so it isn't a controlled comparison on that axis.
- **The budget asymmetry (resolved in this version)**: this note applied when S4D trained for 400 epochs against 15-epoch baselines. As of section 8 above, S4D also trains for `BASELINE_EPOCHS`, so accuracy, generalization-gap, *and* epoch-count are now all directly comparable across all six models. `train_time_s` can still differ for reasons other than epoch count (S4D's augmentation and FFT convolution aren't free), so it's still worth reading that column with the recipe differences in mind — just not an epoch-count asymmetry anymore.

### Bottom line

With the real optimization recipe, S4D earns a legitimate place in this comparison: it matches Transformer's accuracy at roughly a quarter of the parameters and a third the disk size, and it's dramatically more robust to input noise than the CNN that otherwise "wins" on the raw accuracy number. The honest tradeoffs are training cost (a long, specialized schedule that Transformer doesn't need to hit the same accuracy) and inference latency (still the slowest of the six, though halved from the earlier implementation). The strongest version of this justification isn't "S4D is the most accurate model" — it's "S4D gets CNN-adjacent accuracy and beats Transformer's parameter count and noise robustness, which matters specifically for the constrained deployment target (RISC-V/edge) this project is built around."

## 14. Suggested Additional Evaluation Metrics

Beyond what's already measured above (training accuracy, inference time, model
size, macro F1, generalization gap, peak memory, noise robustness):

- **FLOPs / MACs per forward pass** -- hardware-independent, complementary to
  wall-clock latency. `ptflops`/`fvcore` handle CNN/RNN/Transformer directly; S4D's
  FFT convolution needs a manual FLOP-counting hook, since generic tools don't
  recognize custom ops.
- **Peak memory during inference** (not just training) -- relevant for the
  C/RISC-V export path this project also targets.
- **Calibration (Expected Calibration Error)** -- whether predicted confidence
  matches actual accuracy; two models with identical accuracy can differ a lot here.
- **Convergence speed** -- steps/epochs to cross an accuracy threshold, rather than
  final accuracy alone. Directly relevant to S4D given the warm-restart schedule.
- **Per-class precision/recall** -- `sklearn.metrics.classification_report`, useful
  if any one galaxy class matters more than the others for your use case.
- **Sensitivity to Hilbert-order corruption** -- shuffle the curve order and see how
  much accuracy drops, as a direct test of how much S4D/RNN/Transformer actually
  rely on the locality-preserving ordering versus just the pixel values.

In [ ]:
try:
    from ptflops import get_model_complexity_info

    for name, (model, adapter) in nn_model_specs.items():
        macs, params = get_model_complexity_info(model, (1, 64, 64) if name == "CNN" else (64, 64),
                                                   as_strings=True, print_per_layer_stat=False, verbose=False)
        print(f"{name}: {macs} MACs, {params} params")
    print("S4D: skipped -- ptflops doesn't have a hook for the custom FFT convolution op")
except ImportError:
    print("Install `ptflops` (pip install ptflops) to run this cell.")

## 15. Save Comparison Results

In [ ]:
torch.save(fast_model.state_dict(), "model_comparison_outputs/S4D_comparison.pt")
for name, (model, _) in nn_model_specs.items():
    torch.save(model.state_dict(), f"model_comparison_outputs/{name.replace(' ', '_')}_final.pt")

results_df.to_csv("model_comparison_outputs/results_table.csv")
print("Saved all model weights and the results table to model_comparison_outputs/")
print("(S4D here is the BASELINE_EPOCHS comparison checkpoint -- see section 16 "
      "for the full-budget production checkpoint.)")

**Looking for the deployable S4D checkpoint?** `S4D_comparison.pt` saved above
is the `BASELINE_EPOCHS`-budget diagnostic model from this comparison, not the
fully trained one. Section 16, below, trains S4D at its full production budget
and saves that checkpoint separately, along with the recurrent-model conversion
note for the RISC-V export path.

## 16. Final Production Training: S4D at Full Budget

The comparison above puts S4D on the same `BASELINE_EPOCHS` budget as every
other model in the notebook, deliberately, so the accuracy numbers are
comparable. That is not the model you'd actually want to ship -- it's a
diagnostic run, not S4D's best effort.

Now that the comparison confirms S4D is competitive at a matched epoch budget,
this section retrains it from scratch with its full production recipe -- the
same grouped optimizer, warmup, cosine-annealing-with-warm-restarts schedule,
gradient clipping, and rotation/flip augmentation as section 8 (now also
including `S4D_USE_NORM`/`S4D_USE_RESIDUAL`/`S4D_DROPOUT` from section 1, and
whatever section 7.5's Steps 5-6 recommend for `S4D_POOLING`), just without
the `BASELINE_EPOCHS` cap -- to produce the checkpoint you'd actually deploy.

**`S4D_FINAL_EPOCHS` is 630 below, not 400 or the originally-guessed 200 --
here's why, now that an actual run's numbers exist to check against.** The
warm-restart schedule's cycle boundaries, given `T_0=10 epochs, T_mult=2`,
land at epochs 10, 30, 70, 150, 310, and 630. A 400-epoch run stops 90 epochs
into the 310-->630 cycle -- mid-cosine-decay, at a learning rate well above
`eta_min`. An earlier 400-epoch run of this exact recipe reached 80.35% test
accuracy, and its own per-epoch table (Trainer logs epochs 201-400 in the
output below once you run this cell) shows *why* stopping at 400 is an
awkward place to cut it off: validation accuracy peaks at epoch 304 (80.81%)
and then oscillates in the 0.76-0.80 range through epoch 400 with no further
upward trend -- exactly the signature of a run interrupted mid-cycle rather
than one that's actually converged. Letting the schedule complete its next
full cycle (through epoch 630) is the direct fix: ~1.6x the original run's
training time (still well under an hour on a T4), and it answers a real
question that run left open -- does finishing the anneal buy more accuracy,
or does the plateau hold even once LR fully decays? Either answer is useful.
Guessing a *lower* epoch count, as this section originally suggested (200)
before any run's numbers existed to check it against, risks stopping even
earlier mid-cycle, which is the opposite of what the evidence now supports.
If you'd rather spend less time first, 310 is the nearest full-cycle boundary
below 400 -- but epoch 304 of the run described above already sits almost
exactly there, so it mostly re-confirms a number you already have rather than
testing something new.

In [ ]:
# Not a hard gate -- if you're running this notebook top-to-bottom on Colab this
# cell runs either way -- just a clear, printed confirmation that the comparison
# above actually supports spending the full S4D_FINAL_EPOCHS training budget on
# S4D before doing so.
baseline_test_accs = results_df.drop(index="S4D")["test_acc"]
s4d_comparison_acc = results_df.loc["S4D", "test_acc"]

print(f"S4D ({S4D_COMPARISON_EPOCHS}-epoch comparison run): test_acc = {s4d_comparison_acc:.4f}")
print(f"Baselines ({S4D_COMPARISON_EPOCHS}-epoch run):        "
      f"best = {baseline_test_accs.max():.4f}  mean = {baseline_test_accs.mean():.4f}")

if s4d_comparison_acc >= baseline_test_accs.mean():
    print("\nS4D performs at least as well as the average baseline at a matched "
          "epoch budget -- proceeding to the full production training run below.")
else:
    print("\nS4D is currently below the average baseline at a matched epoch budget. "
          "The cells below will still run the full recipe, but it's worth reviewing "
          "the comparison results above before spending the extra training time.")

### 16.1 Setup

Reuses the same Drive mount and `CHECKPOINT_DIR` from section 8 -- just a new
subdirectory, since this run's checkpoints (`S4D_FINAL_EPOCHS` epochs, a
different LR schedule) aren't compatible with the comparison run's.

In [ ]:
CHECKPOINT_DIR_FINAL = os.path.join(
    os.path.dirname(CHECKPOINT_DIR), f"GalaxyS4_Checkpoints_final_{s4d_config_tag}"
)
os.makedirs(CHECKPOINT_DIR_FINAL, exist_ok=True)
print(f"Final production checkpoints will be saved to:\n  {CHECKPOINT_DIR_FINAL}")

### 16.2 Train

Fresh model (same `S4D_STATE`/`S4D_NUM_LAYERS`/`S4D_PATCH_SIZE`/`S4D_POOLING`
config as section 8), same seed, same augmented datasets -- only the epoch
count and the warm-restart schedule's `T_0` go back to the full recipe's
original scale (`T_0` = 10 epochs' worth of steps), rather than the scaled-down
value used for the short comparison run. This is the long pole in the
notebook -- budget real time for it. Checkpoints save every epoch and resume
automatically if this cell is interrupted and re-run -- safe to stop early.

In [ ]:
torch.manual_seed(RNG_SEED)
final_model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE, d_model=S4D_STATE, num_classes=NUM_CLASSES, colored=COLORED,
    num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE, pooling=S4D_POOLING,
    use_norm=S4D_USE_NORM, use_residual=S4D_USE_RESIDUAL, dropout=S4D_DROPOUT,
    patch_embed=S4D_PATCH_EMBED
).to(DEVICE)
final_wrapped_model = HF_GalaxyModelWrapper(final_model)

S4D_FINAL_EPOCHS = 700  # was 630 -- warmup (63 epochs' worth of steps) pushes the
                          # restart scheduler's boundaries 63 epochs later than their
                          # nominal values, so the real final cycle is 373->693, not
                          # ...->630. Val acc was still setting new highs at epoch 628,
                          # not flat like the last run's tail was.

final_optimizer = build_grouped_optimizer(final_wrapped_model, lr=1e-3, weight_decay=0.05)

# Linear warmup (10% of total steps) into cosine annealing with warm restarts,
# back to the full recipe's original schedule scale: T_0=10 epochs' worth of
# steps, T_mult=2 -- restarts at epoch 10, 30, 70, 150, 310, with the 6th cycle
# (310-->630) now finishing its anneal exactly at S4D_FINAL_EPOCHS instead of
# being cut off 90 epochs in, which is what a 400-epoch budget did here before.
final_warmup_steps = max(1, int(0.1 * S4D_FINAL_EPOCHS * s4d_steps_per_epoch))
final_warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    final_optimizer, start_factor=0.1, total_iters=final_warmup_steps
)
final_restart_scheduler = CosineAnnealingWarmRestarts(
    final_optimizer, T_0=10 * s4d_steps_per_epoch, T_mult=2, eta_min=1e-5
)
final_scheduler = torch.optim.lr_scheduler.SequentialLR(
    final_optimizer, schedulers=[final_warmup_scheduler, final_restart_scheduler],
    milestones=[final_warmup_steps]
)

final_training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR_FINAL,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    logging_steps=50,
    learning_rate=1e-3,
    per_device_train_batch_size=S4D_BATCH_SIZE,
    per_device_eval_batch_size=S4D_BATCH_SIZE,
    num_train_epochs=S4D_FINAL_EPOCHS,
    weight_decay=0.05,               # enforced per-group by build_grouped_optimizer above
    max_grad_norm=1.0,               # gradient clipping -- matters more at this larger capacity
    lr_scheduler_type="constant",    # ours (warmup + warm restarts) drives LR instead
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    dataloader_num_workers=2,
    dataloader_persistent_workers=True,
    push_to_hub=False,
    report_to="none",
)

final_trainer = Trainer(
    model=final_wrapped_model,
    args=final_training_args,
    train_dataset=train_dataset,   # same GalaxyDatasetAug instances from section 8
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    optimizers=(final_optimizer, final_scheduler),
)

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
final_start = time.time()

last_final_checkpoint = get_last_checkpoint(CHECKPOINT_DIR_FINAL) if os.path.isdir(CHECKPOINT_DIR_FINAL) else None
final_trainer.train(resume_from_checkpoint=last_final_checkpoint)

final_train_time = time.time() - final_start
final_peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if DEVICE == "cuda" else float("nan")

final_model.eval()
print(f"\nFinal production training complete in {final_train_time:.1f}s ({S4D_FINAL_EPOCHS} epochs)")

### 16.3 Evaluate the Production Model

Same evaluation protocol as everywhere else in this notebook (section 6), plus
the same test-time-augmentation check from section 8.1, run against the fully
trained model instead of the comparison-run one.

In [ ]:
final_for_eval = LogitsAdapter(final_model)
final_result, (final_y_true, final_y_pred) = collect_model_metrics(
    final_for_eval, adapt_s4d, "S4D (production)", final_train_time, final_peak_mem_mb,
    "model_comparison_outputs/S4D_production.pt"
)

preds_scans, _ = predict_with_tta_and_scans(final_model, X_test, DEVICE, use_flips=False, batch_size=256)
acc_scans = (preds_scans.cpu() == y_test.cpu()).float().mean().item()

preds_tta, _ = predict_with_tta_and_scans(final_model, X_test, DEVICE, use_flips=True, batch_size=256)
acc_tta = (preds_tta.cpu() == y_test.cpu()).float().mean().item()

#print(f"S4D, {S4D_COMPARISON_EPOCHS}-epoch comparison run:  test_acc = {s4d_comparison_acc:.4f}")
print(f"S4D, {S4D_FINAL_EPOCHS}-epoch production run:       test_acc = {final_result['test_acc']:.4f}  "
      f"macro_f1 = {final_result['test_macro_f1']:.4f}")
print(f"S4D production, multi-scan (4 orient.):  {acc_scans:.4f}")
print(f"S4D production, TTA (8 orientations):    {acc_tta:.4f}")

### 16.4 Full Statistics Table (Production Run)

`final_result` (16.3) was already built by `collect_model_metrics` -- the same
function, same schema, as every row in section 10's `results_df` -- it just
was never displayed as its own table. This is that table: params, size,
latency, throughput, memory, everything, for the actual `S4D_FINAL_EPOCHS`-budget
checkpoint instead of the `BASELINE_EPOCHS` comparison-run row.

Also concatenated onto `results_df` below for convenience, clearly as an extra
row rather than merged in above -- section 10's note already covers why: this
row trained for `S4D_FINAL_EPOCHS` epochs with its own recipe, not the shared
`BASELINE_EPOCHS` budget the rest of that table is matched on, so `train_time_s`
and `generalization_gap` in particular aren't an apples-to-apples comparison
against the other ten rows the way they are against each other.

In [ ]:
final_stats_df = pd.DataFrame([final_result]).set_index("model")
final_stats_df = final_stats_df.round(4)
final_stats_df.to_csv("model_comparison_outputs/S4D_production_stats.csv")
final_stats_df

In [ ]:
# Same eleven rows as results_df, plus this one -- S4D (production) labeled
# distinctly from S4D's own BASELINE_EPOCHS-budget row so both stay visible.
full_comparison_df = pd.concat([results_df, final_stats_df])
full_comparison_df.to_csv("model_comparison_outputs/full_comparison_table.csv")
full_comparison_df

### 16.5 Save the Production Model

This is the checkpoint worth keeping -- trained with the full recipe, not the
`BASELINE_EPOCHS` diagnostic budget used for the comparison above.

In [ ]:
torch.save(final_model.state_dict(), "model_comparison_outputs/S4D_production_finalv2.pt")
print("Saved the full-budget production S4D checkpoint to "
      "model_comparison_outputs/S4D_production_finalv2.pt")

**About porting this back to the RISC-V recurrent model:** the state-dict
compatibility the earlier version of this notebook relied on (`GalaxyClassifierS4DFast`
and `model.gclassifier.GalaxyClassifierS4D` sharing parameter names/shapes) no
longer holds -- `final_model` has patch embedding, mean pooling, and
`S4D_NUM_LAYERS` layers, none of which the recurrent model implements. A plain

```python
from model.gclassifier import GalaxyClassifierS4D
recurrent_model = GalaxyClassifierS4D(s4_state=64, d_model=64, num_classes=NUM_CLASSES, colored=COLORED)
recurrent_model.load_state_dict(final_model.state_dict())  # will raise -- shapes/keys no longer match
```

will fail. Porting these accuracy gains to the RISC-V deployment path for real
means updating `model/hilbert.py` (patch-based scan), `model/s4d_recurrent.py`
(the extra layer, matching dims) and `model/gclassifier.py` (mean pooling) to
mirror what this notebook does -- a separate, larger task outside what this
comparison notebook does on its own. If deployability matters more than the
accuracy gain for your use case, `S4D_PATCH_SIZE=1`, `S4D_POOLING="last"`,
`S4D_NUM_LAYERS=2` and `S4D_STATE=64` recover the original, already-portable
architecture -- change those four values back and every cell in this notebook
still runs, just training the old model instead.

In [ ]:
# History extraction -- same pattern section 8 uses for histories["S4D"], just
# pointed at final_trainer instead of s4d_trainer.
final_log_epochs, final_log_loss = [], []
final_eval_epochs, final_eval_acc = [], []
for log in final_trainer.state.log_history:
    if "loss" in log:
        final_log_epochs.append(log["epoch"])
        final_log_loss.append(log["loss"])
    if "eval_accuracy" in log:
        final_eval_epochs.append(log["epoch"])
        final_eval_acc.append(log["eval_accuracy"])

histories["S4D (production)"] = {"epochs": final_log_epochs, "loss": final_log_loss,
                                  "eval_epochs": final_eval_epochs, "eval_acc": final_eval_acc}

In [ ]:
# Loss + validation accuracy curves -- FULL S4D_FINAL_EPOCHS run
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(histories["S4D (production)"]["epochs"], histories["S4D (production)"]["loss"],
             color="seagreen")
axes[0].set_title(f"S4D Training Loss -- FULL {S4D_FINAL_EPOCHS}-EPOCH PRODUCTION RUN")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(histories["S4D (production)"]["eval_epochs"], histories["S4D (production)"]["eval_acc"],
             color="seagreen", marker="o")
axes[1].set_title(f"S4D Validation Accuracy -- FULL {S4D_FINAL_EPOCHS}-EPOCH PRODUCTION RUN")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for the production model
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(final_y_true, final_y_pred, labels=range(NUM_CLASSES))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
ax.set_title(f"S4D production ({S4D_FINAL_EPOCHS} epochs) -- test acc: {final_result['test_acc']:.2%}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.show()